In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import gc
import shutil
import traceback
import cv2

from tqdm.auto import tqdm

print("=" * 70)
print("PHASE 6 — SHARDED CLIP GENERATION")
print("=" * 70)
print("Imports ready")

PHASE 6 — SHARDED CLIP GENERATION
Imports ready


In [2]:
# ============================================================
# PHASE 6 — CELL 2
# Project Configuration
# ============================================================

PROJECT_ROOT = Path(
    r"C:\LipReadingSSL"
).resolve()

OUTPUT_ROOT = (
    PROJECT_ROOT /
    "output"
)

RAW_VIDEO_DIR = (
    PROJECT_ROOT /
    "dataset" /
    "raw_videos"
)

# ------------------------------------------------------------
# Videos
# ------------------------------------------------------------

VIDEO_IDS = [
    "video001",
    "video002",
    "video003",
]

# ------------------------------------------------------------
# Phase 5 folders / files
# ------------------------------------------------------------

MOUTH_CROP_DIR_NAME = "mouth_crop"

MOUTH_METADATA_NAME = (
    "mouth_metadata.csv"
)

# ------------------------------------------------------------
# Phase 6 output
# ------------------------------------------------------------

CLIPS_DIR_NAME = "clips"

CLIP_NPY_DIR_NAME = "clip_npy"

CLIP_METADATA_NAME = (
    "clip_metadata.csv"
)

PHASE6_STATE_NAME = (
    "phase6_state.json"
)

PHASE6_LOG_NAME = (
    "phase6_log.csv"
)

# ------------------------------------------------------------
# Clip settings
# ------------------------------------------------------------

CLIP_LENGTH = 16

CLIP_STRIDE = 1

MOUTH_HEIGHT = 96
MOUTH_WIDTH = 96

# ------------------------------------------------------------
# Sharding
# ------------------------------------------------------------

SHARD_SIZE = 512

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

SAVE_JPG_CLIPS = False

SAVE_NPY_CLIPS = False

SAVE_SHARDS = True

# ------------------------------------------------------------
# Resume
# ------------------------------------------------------------

RESUME = True

# ------------------------------------------------------------
# Compression
# ------------------------------------------------------------

SHARD_COMPRESSED = True

# ------------------------------------------------------------
# Data type
# uint8 = 1 byte / pixel
# ------------------------------------------------------------

CLIP_DTYPE = np.uint8

print()
print(f"Project root : {PROJECT_ROOT}")
print(f"Output root  : {OUTPUT_ROOT}")
print(f"Clip length  : {CLIP_LENGTH}")
print(f"Clip stride  : {CLIP_STRIDE}")
print(f"Shard size   : {SHARD_SIZE}")
print(f"Clip dtype   : {CLIP_DTYPE}")
print("=" * 70)


Project root : C:\LipReadingSSL
Output root  : C:\LipReadingSSL\output
Clip length  : 16
Clip stride  : 1
Shard size   : 512
Clip dtype   : <class 'numpy.uint8'>


In [3]:
# ============================================================
# PHASE 6 — CELL 3
# Phase 5 Path Resolver
# ============================================================

def get_phase6_paths(video_id):

    video_id = str(video_id)

    video_output_dir = (
        OUTPUT_ROOT /
        video_id
    )

    mouth_crop_dir = (
        video_output_dir /
        MOUTH_CROP_DIR_NAME
    )

    mouth_metadata = (
        video_output_dir /
        MOUTH_METADATA_NAME
    )

    clips_dir = (
        video_output_dir /
        CLIPS_DIR_NAME
    )

    clip_npy_dir = (
        video_output_dir /
        CLIP_NPY_DIR_NAME
    )

    phase6_state = (
        video_output_dir /
        PHASE6_STATE_NAME
    )

    phase6_log = (
        video_output_dir /
        PHASE6_LOG_NAME
    )

    clip_metadata = (
        video_output_dir /
        CLIP_METADATA_NAME
    )

    return {
        "video_output_dir":
            video_output_dir,

        "mouth_crop":
            mouth_crop_dir,

        "mouth_metadata":
            mouth_metadata,

        "clips":
            clips_dir,

        "clip_npy":
            clip_npy_dir,

        "clip_metadata":
            clip_metadata,

        "phase6_state":
            phase6_state,

        "phase6_log":
            phase6_log,
    }


print("Phase 6 path resolver ready")

Phase 6 path resolver ready


In [4]:
# ============================================================
# PHASE 6 — CELL 4
# Output Structure
# ============================================================

def create_phase6_output_structure(video_id):

    paths = get_phase6_paths(
        video_id
    )

    required_dirs = [
        paths["video_output_dir"],
        paths["clips"],
        paths["clip_npy"],
    ]

    for directory in required_dirs:

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    return paths


print("Output structure ready")

Output structure ready


In [5]:
# ============================================================
# PHASE 6 — CELL 5
# Frame ID Utilities
# ============================================================

FRAME_PATTERN = re.compile(
    r"frame_(\d+)"
)


def extract_frame_index(path):

    path = Path(path)

    match = FRAME_PATTERN.search(
        path.stem
    )

    if match is None:
        return None

    return int(
        match.group(1)
    )


def discover_mouth_frames(
    mouth_dir
):

    mouth_dir = Path(
        mouth_dir
    )

    frame_map = {}

    if not mouth_dir.exists():
        return frame_map

    for path in mouth_dir.iterdir():

        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp",
        }:
            continue

        frame_index = (
            extract_frame_index(path)
        )

        if frame_index is None:
            continue

        frame_map[
            frame_index
        ] = path

    return frame_map


print("Frame utilities ready")

Frame utilities ready


In [6]:
# ============================================================
# PHASE 6 — CELL 6
# Read Phase 5 Metadata
# ============================================================

def load_mouth_metadata(
    video_id,
    paths
):

    metadata_path = (
        paths["mouth_metadata"]
    )

    if not metadata_path.exists():

        raise FileNotFoundError(
            f"Phase 5 metadata not found:\n"
            f"{metadata_path}"
        )

    df = pd.read_csv(
        metadata_path
    )

    if df.empty:

        raise RuntimeError(
            f"Phase 5 metadata is empty:\n"
            f"{metadata_path}"
        )

    if "frame_index" not in df.columns:

        raise ValueError(
            "Phase 5 metadata missing "
            "'frame_index'"
        )

    df["frame_index"] = pd.to_numeric(
        df["frame_index"],
        errors="coerce"
    )

    df = df[
        df["frame_index"].notna()
    ].copy()

    df["frame_index"] = (
        df["frame_index"]
        .astype(int)
    )

    df = (
        df
        .drop_duplicates(
            subset=["frame_index"],
            keep="last"
        )
        .sort_values(
            "frame_index"
        )
        .reset_index(drop=True)
    )

    if "video_id" in df.columns:

        df = df[
            df["video_id"].astype(str)
            == str(video_id)
        ].copy()

    return df


print("Metadata loader ready")

Metadata loader ready


In [7]:
# ============================================================
# PHASE 6 — CELL 7
# Build Continuous Sequences
# ============================================================

def build_continuous_sequences(
    frame_ids,
    clip_length,
    stride
):

    frame_ids = sorted(
        set(
            int(x)
            for x in frame_ids
        )
    )

    if len(frame_ids) < clip_length:

        return []

    frame_set = set(
        frame_ids
    )

    sequences = []

    for start_pos in range(
        0,
        len(frame_ids) - clip_length + 1,
        stride
    ):

        sequence = frame_ids[
            start_pos:
            start_pos + clip_length
        ]

        expected = list(
            range(
                sequence[0],
                sequence[0] +
                clip_length
            )
        )

        if sequence != expected:
            continue

        if not all(
            frame_id in frame_set
            for frame_id in expected
        ):
            continue

        sequences.append(
            sequence
        )

    return sequences


print("Continuous sequence builder ready")

Continuous sequence builder ready


In [8]:
# ============================================================
# PHASE 6 — CELL 8
# Load Mouth Frame
# ============================================================

def load_mouth_frame(
    frame_path
):

    frame_path = Path(
        frame_path
    )

    image = cv2.imread(
        str(frame_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:

        raise RuntimeError(
            f"Cannot read mouth frame:\n"
            f"{frame_path}"
        )

    if (
        image.shape[0] != MOUTH_HEIGHT
        or
        image.shape[1] != MOUTH_WIDTH
    ):

        image = cv2.resize(
            image,
            (
                MOUTH_WIDTH,
                MOUTH_HEIGHT
            ),
            interpolation=cv2.INTER_AREA
        )

    return image.astype(
        CLIP_DTYPE,
        copy=False
    )


print("Mouth frame loader ready")

Mouth frame loader ready


In [9]:
# ============================================================
# PHASE 6 — CELL 9
# Build Clip Array
# ============================================================

def build_clip_array(
    frame_sequence,
    mouth_frame_map
):

    clip = np.empty(
        (
            len(frame_sequence),
            MOUTH_HEIGHT,
            MOUTH_WIDTH
        ),
        dtype=CLIP_DTYPE
    )

    for i, frame_index in enumerate(
        frame_sequence
    ):

        frame_path = (
            mouth_frame_map[
                frame_index
            ]
        )

        clip[i] = load_mouth_frame(
            frame_path
        )

    return clip

In [10]:
# ============================================================
# PHASE 6 — CELL 10
# Shard Writer
# ============================================================

def save_shard(
    clips,
    clip_records,
    shard_path
):

    shard_path = Path(
        shard_path
    )

    shard_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    data = np.stack(
        clips,
        axis=0
    )

    starts = np.array(
        [
            record["start_frame"]
            for record in clip_records
        ],
        dtype=np.int32
    )

    ends = np.array(
        [
            record["end_frame"]
            for record in clip_records
        ],
        dtype=np.int32
    )

    indices = np.arange(
        len(clip_records),
        dtype=np.int32
    )

    if SHARD_COMPRESSED:

        np.savez_compressed(
            shard_path,
            clips=data,
            start_frames=starts,
            end_frames=ends,
            clip_indices=indices
        )

    else:

        np.savez(
            shard_path,
            clips=data,
            start_frames=starts,
            end_frames=ends,
            clip_indices=indices
        )

    del data

    gc.collect()


print("Shard writer ready")

Shard writer ready


In [11]:
# ============================================================
# PHASE 6 — CELL 11
# Process One Video
# ============================================================

def process_video(
    video_id
):

    video_id = str(
        video_id
    )

    paths = (
        create_phase6_output_structure(
            video_id
        )
    )

    print()
    print("=" * 70)
    print(
        f"PHASE 6 PROCESSING : {video_id}"
    )
    print("=" * 70)

    print(
        f"Mouth crop : "
        f"{paths['mouth_crop']}"
    )

    print(
        f"Metadata   : "
        f"{paths['mouth_metadata']}"
    )

    print(
        f"Shard dir  : "
        f"{paths['clip_npy']}"
    )

    # --------------------------------------------------------
    # Load mouth frame map
    # --------------------------------------------------------

    mouth_frame_map = (
        discover_mouth_frames(
            paths["mouth_crop"]
        )
    )

    if not mouth_frame_map:

        raise RuntimeError(
            "No mouth crop frames found"
        )

    # --------------------------------------------------------
    # Load metadata
    # --------------------------------------------------------

    metadata_df = (
        load_mouth_metadata(
            video_id,
            paths
        )
    )

    metadata_frame_ids = set(
        metadata_df[
            "frame_index"
        ].tolist()
    )

    # --------------------------------------------------------
    # Use intersection
    # --------------------------------------------------------

    valid_frame_ids = (
        set(
            mouth_frame_map.keys()
        )
        &
        metadata_frame_ids
    )

    print()
    print(
        f"Metadata frames : "
        f"{len(metadata_frame_ids)}"
    )

    print(
        f"Mouth frames    : "
        f"{len(mouth_frame_map)}"
    )

    print(
        f"Usable frames   : "
        f"{len(valid_frame_ids)}"
    )

    # --------------------------------------------------------
    # Continuous sequences
    # --------------------------------------------------------

    sequences = (
        build_continuous_sequences(
            valid_frame_ids,
            CLIP_LENGTH,
            CLIP_STRIDE
        )
    )

    total_clips = len(
        sequences
    )

    print(
        f"Generated clips : "
        f"{total_clips}"
    )

    if total_clips == 0:

        raise RuntimeError(
            "No continuous clips can be generated"
        )

    # --------------------------------------------------------
    # Existing metadata
    # --------------------------------------------------------

    existing_records = []

    if (
        RESUME
        and
        paths["clip_metadata"].exists()
    ):

        try:

            old_df = pd.read_csv(
                paths["clip_metadata"]
            )

            existing_records = (
                old_df.to_dict(
                    "records"
                )
            )

        except Exception:

            existing_records = []

    existing_keys = {
        (
            int(r["start_frame"]),
            int(r["end_frame"])
        )
        for r in existing_records
        if
        "start_frame" in r
        and
        "end_frame" in r
    }

    # --------------------------------------------------------
    # Shard buffers
    # --------------------------------------------------------

    shard_clips = []
    shard_records = []

    all_records = list(
        existing_records
    )

    generated = 0
    skipped = 0
    shard_id = 0

    # --------------------------------------------------------
    # Existing shard detection
    # --------------------------------------------------------

    existing_shards = sorted(
        paths["clip_npy"].glob(
            "shard_*.npz"
        )
    )

    if existing_shards:

        shard_numbers = []

        for path in existing_shards:

            match = re.search(
                r"shard_(\d+)",
                path.stem
            )

            if match:

                shard_numbers.append(
                    int(match.group(1))
                )

        if shard_numbers:

            shard_id = (
                max(shard_numbers) + 1
            )

    # --------------------------------------------------------
    # Process sequences
    # --------------------------------------------------------

    progress = tqdm(
        sequences,
        desc=f"{video_id}",
        unit="clip"
    )

    for sequence in progress:

        start_frame = int(
            sequence[0]
        )

        end_frame = int(
            sequence[-1]
        )

        key = (
            start_frame,
            end_frame
        )

        if key in existing_keys:

            skipped += 1
            continue

        try:

            clip = build_clip_array(
                sequence,
                mouth_frame_map
            )

        except Exception as e:

            print()
            print(
                f"Skip clip "
                f"{start_frame}-{end_frame}: "
                f"{type(e).__name__}"
            )

            skipped += 1
            continue

        record = {

            "video_id":
                video_id,

            "clip_index":
                len(all_records),

            "start_frame":
                start_frame,

            "end_frame":
                end_frame,

            "num_frames":
                CLIP_LENGTH,

            "shard_id":
                shard_id,

            "dtype":
                str(CLIP_DTYPE),

            "height":
                MOUTH_HEIGHT,

            "width":
                MOUTH_WIDTH,

            "storage":
                "shard_npz",
        }

        shard_clips.append(
            clip
        )

        shard_records.append(
            record
        )

        all_records.append(
            record
        )

        generated += 1

        # ----------------------------------------------------
        # Flush shard
        # ----------------------------------------------------

        if len(shard_clips) >= SHARD_SIZE:

            shard_path = (
                paths["clip_npy"] /
                f"shard_{shard_id:06d}.npz"
            )

            save_shard(
                shard_clips,
                shard_records,
                shard_path
            )

            shard_clips = []
            shard_records = []

            shard_id += 1

            gc.collect()

    # --------------------------------------------------------
    # Flush final shard
    # --------------------------------------------------------

    if shard_clips:

        shard_path = (
            paths["clip_npy"] /
            f"shard_{shard_id:06d}.npz"
        )

        save_shard(
            shard_clips,
            shard_records,
            shard_path
        )

        shard_clips = []
        shard_records = []

        gc.collect()

    # --------------------------------------------------------
    # Save clip metadata
    # --------------------------------------------------------

    result_df = pd.DataFrame(
        all_records
    )

    if not result_df.empty:

        result_df = (
            result_df
            .drop_duplicates(
                subset=[
                    "video_id",
                    "start_frame",
                    "end_frame"
                ],
                keep="last"
            )
            .sort_values(
                [
                    "video_id",
                    "start_frame"
                ]
            )
            .reset_index(
                drop=True
            )
        )

        result_df[
            "clip_index"
        ] = np.arange(
            len(result_df),
            dtype=np.int64
        )

        result_df.to_csv(
            paths["clip_metadata"],
            index=False,
            encoding="utf-8-sig"
        )

    # --------------------------------------------------------
    # State
    # --------------------------------------------------------

    state = {

        "video_id":
            video_id,

        "status":
            "COMPLETED",

        "mouth_frames":
            len(mouth_frame_map),

        "metadata_frames":
            len(metadata_frame_ids),

        "usable_frames":
            len(valid_frame_ids),

        "total_clips":
            total_clips,

        "generated":
            generated,

        "skipped_existing":
            skipped,

        "clip_length":
            CLIP_LENGTH,

        "clip_stride":
            CLIP_STRIDE,

        "shard_size":
            SHARD_SIZE,

        "storage":
            "shard_npz",

        "clip_metadata":
            str(
                paths["clip_metadata"]
            ),
    }

    with open(
        paths["phase6_state"],
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            indent=2,
            ensure_ascii=False
        )

    print()
    print("=" * 70)
    print(
        f"PHASE 6 COMPLETED : {video_id}"
    )
    print("=" * 70)

    print(
        f"Mouth frames    : "
        f"{len(mouth_frame_map)}"
    )

    print(
        f"Usable frames   : "
        f"{len(valid_frame_ids)}"
    )

    print(
        f"Possible clips  : "
        f"{total_clips}"
    )

    print(
        f"Generated       : "
        f"{generated}"
    )

    print(
        f"Skipped         : "
        f"{skipped}"
    )

    print(
        f"Clip metadata   : "
        f"{paths['clip_metadata']}"
    )

    print(
        f"Shard directory : "
        f"{paths['clip_npy']}"
    )

    print("=" * 70)

    return state

In [12]:
# ============================================================
# PHASE 6 — CELL 12
# Batch Runner + Full Error Debug
# ============================================================

PHASE6_SUMMARY = []

print()
print("=" * 70)
print("PHASE 6 — SHARDED CLIP GENERATION")
print("=" * 70)
print(f"Videos: {len(VIDEO_IDS)}")

for position, video_id in enumerate(
    VIDEO_IDS,
    start=1
):
    print()
    print("=" * 70)
    print(
        f"[{position}/{len(VIDEO_IDS)}] "
        f"{video_id}"
    )
    print("=" * 70)

    try:
        state = process_video(
            video_id
        )

        PHASE6_SUMMARY.append({
            "video_id":
                video_id,

            "status":
                "COMPLETED",

            "clips":
                state.get(
                    "total_clips",
                    0
                ),

            "generated":
                state.get(
                    "generated",
                    0
                ),

            "skipped":
                state.get(
                    "skipped_existing",
                    0
                ),

            "error":
                None
        })

    except Exception as e:

        # ----------------------------------------------------
        # IMPORTANT:
        # Do NOT hide the original exception.
        # Print the actual NameError variable.
        # ----------------------------------------------------

        print()
        print("!" * 70)
        print(
            f"❌ FAILED: {video_id}"
        )
        print("!" * 70)

        print(
            f"Error type : {type(e).__name__}"
        )

        print(
            f"Error      : {e}"
        )

        print()
        print(
            "FULL TRACEBACK"
        )
        print("-" * 70)

        traceback.print_exc()

        print("-" * 70)
        print(
            "The error above is the REAL error "
            "that caused this clip to fail."
        )
        print("-" * 70)

        PHASE6_SUMMARY.append({
            "video_id":
                video_id,

            "status":
                "FAILED",

            "clips":
                0,

            "generated":
                0,

            "skipped":
                0,

            "error":
                (
                    f"{type(e).__name__}: "
                    f"{e}"
                )
        })

        # ----------------------------------------------------
        # Stop immediately.
        #
        # This is intentional for debugging.
        # Otherwise Phase 6 will continue processing
        # thousands of clips with the same error.
        # ----------------------------------------------------

        raise

    finally:
        gc.collect()


# ============================================================
# Summary
# ============================================================

PHASE6_SUMMARY_DF = pd.DataFrame(
    PHASE6_SUMMARY
)

print()
print("=" * 70)
print("PHASE 6 — BATCH SUMMARY")
print("=" * 70)

if not PHASE6_SUMMARY_DF.empty:
    print(
        PHASE6_SUMMARY_DF.to_string(
            index=False
        )
    )

print("=" * 70)


PHASE 6 — SHARDED CLIP GENERATION
Videos: 3

[1/3] video001

PHASE 6 PROCESSING : video001
Mouth crop : C:\LipReadingSSL\output\video001\mouth_crop
Metadata   : C:\LipReadingSSL\output\video001\mouth_metadata.csv
Shard dir  : C:\LipReadingSSL\output\video001\clip_npy

Metadata frames : 41058
Mouth frames    : 40746
Usable frames   : 40746
Generated clips : 40076


video001:   0%|          | 0/40076 [00:00<?, ?clip/s]


PHASE 6 COMPLETED : video001
Mouth frames    : 40746
Usable frames   : 40746
Possible clips  : 40076
Generated       : 40076
Skipped         : 0
Clip metadata   : C:\LipReadingSSL\output\video001\clip_metadata.csv
Shard directory : C:\LipReadingSSL\output\video001\clip_npy

[2/3] video002

PHASE 6 PROCESSING : video002
Mouth crop : C:\LipReadingSSL\output\video002\mouth_crop
Metadata   : C:\LipReadingSSL\output\video002\mouth_metadata.csv
Shard dir  : C:\LipReadingSSL\output\video002\clip_npy

Metadata frames : 34794
Mouth frames    : 34278
Usable frames   : 34278
Generated clips : 31662


video002:   0%|          | 0/31662 [00:00<?, ?clip/s]


PHASE 6 COMPLETED : video002
Mouth frames    : 34278
Usable frames   : 34278
Possible clips  : 31662
Generated       : 31662
Skipped         : 0
Clip metadata   : C:\LipReadingSSL\output\video002\clip_metadata.csv
Shard directory : C:\LipReadingSSL\output\video002\clip_npy

[3/3] video003

PHASE 6 PROCESSING : video003
Mouth crop : C:\LipReadingSSL\output\video003\mouth_crop
Metadata   : C:\LipReadingSSL\output\video003\mouth_metadata.csv
Shard dir  : C:\LipReadingSSL\output\video003\clip_npy

Metadata frames : 28276
Mouth frames    : 28110
Usable frames   : 28110
Generated clips : 26273


video003:   0%|          | 0/26273 [00:00<?, ?clip/s]


PHASE 6 COMPLETED : video003
Mouth frames    : 28110
Usable frames   : 28110
Possible clips  : 26273
Generated       : 26273
Skipped         : 0
Clip metadata   : C:\LipReadingSSL\output\video003\clip_metadata.csv
Shard directory : C:\LipReadingSSL\output\video003\clip_npy

PHASE 6 — BATCH SUMMARY
video_id    status  clips  generated  skipped error
video001 COMPLETED  40076      40076        0  None
video002 COMPLETED  31662      31662        0  None
video003 COMPLETED  26273      26273        0  None


In [13]:
# ============================================================
# PHASE 6 — CELL 13
# Validation Configuration + Paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import traceback
import json


# ============================================================
# Project
# ============================================================

PROJECT_ROOT = Path(
    r"C:\LipReadingSSL"
).resolve()

OUTPUT_ROOT = (
    PROJECT_ROOT /
    "output"
)


# ============================================================
# Videos
# ============================================================

VIDEO_IDS = [
    "video001",
    "video002",
    "video003",
]


# ============================================================
# Phase 6 output
# ============================================================

CLIP_METADATA_NAME = (
    "clip_metadata.csv"
)

CLIP_SHARD_DIR_NAME = (
    "clip_npy"
)


# ============================================================
# Expected clip configuration
# ============================================================

CLIP_LENGTH = 16


# ============================================================
# Validation limits
#
# We do NOT load all clips into memory.
# Only a small number of shards/clips are inspected.
# ============================================================

SAMPLE_SHARDS = 3
SAMPLE_CLIPS_PER_SHARD = 2


# ============================================================
# Helpers
# ============================================================

def get_phase6_paths(video_id):

    video_dir = (
        OUTPUT_ROOT /
        video_id
    )

    return {
        "output_dir":
            video_dir,

        "metadata":
            video_dir /
            CLIP_METADATA_NAME,

        "shard_dir":
            video_dir /
            CLIP_SHARD_DIR_NAME,
    }


print("=" * 70)
print("PHASE 6 — VALIDATION CONFIGURATION")
print("=" * 70)

print(
    f"Project root : {PROJECT_ROOT}"
)

print(
    f"Output root  : {OUTPUT_ROOT}"
)

print(
    f"Videos       : {len(VIDEO_IDS)}"
)

print(
    f"Clip length  : {CLIP_LENGTH}"
)

print("=" * 70)

PHASE 6 — VALIDATION CONFIGURATION
Project root : C:\LipReadingSSL
Output root  : C:\LipReadingSSL\output
Videos       : 3
Clip length  : 16


In [14]:
# ============================================================
# PHASE 6 — CELL 14
# Per-Video Validation
# ============================================================

def validate_phase6_video(video_id):

    paths = get_phase6_paths(
        video_id
    )

    result = {
        "video_id":
            video_id,

        "metadata_exists":
            False,

        "shard_dir_exists":
            False,

        "metadata_rows":
            0,

        "shard_files":
            0,

        "shard_clips":
            0,

        "metadata_duplicates":
            0,

        "metadata_missing_columns":
            0,

        "shard_read_errors":
            0,

        "sample_shape_errors":
            0,

        "sample_dtype_errors":
            0,

        "sample_empty_shards":
            0,

        "valid":
            False,

        "errors":
            []
    }


    print()
    print("=" * 70)
    print(
        f"PHASE 6 VALIDATION : {video_id}"
    )
    print("=" * 70)

    print(
        f"Metadata : {paths['metadata']}"
    )

    print(
        f"Shards   : {paths['shard_dir']}"
    )


    # ========================================================
    # Metadata existence
    # ========================================================

    if not paths["metadata"].exists():

        result["errors"].append(
            "missing_clip_metadata"
        )

        print(
            "❌ clip_metadata.csv not found"
        )

        return result

    result[
        "metadata_exists"
    ] = True


    # ========================================================
    # Shard directory
    # ========================================================

    if not paths["shard_dir"].exists():

        result["errors"].append(
            "missing_shard_directory"
        )

        print(
            "❌ clip_npy directory not found"
        )

        return result

    result[
        "shard_dir_exists"
    ] = True


    # ========================================================
    # Read metadata
    # ========================================================

    try:

        metadata_df = pd.read_csv(
            paths["metadata"]
        )

    except Exception as e:

        result["errors"].append(
            "metadata_read_error"
        )

        print(
            f"❌ Metadata read error: "
            f"{type(e).__name__}: {e}"
        )

        return result


    result[
        "metadata_rows"
    ] = len(metadata_df)


    print(
        f"Metadata rows : "
        f"{len(metadata_df)}"
    )


    # ========================================================
    # Metadata columns
    # ========================================================

    required_columns = [
        "video_id",
        "start_frame",
        "end_frame",
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in metadata_df.columns
    ]

    result[
        "metadata_missing_columns"
    ] = len(
        missing_columns
    )

    if missing_columns:

        result["errors"].append(
            "metadata_missing_columns"
        )

        print(
            "❌ Missing metadata columns:",
            missing_columns
        )

        return result


    # ========================================================
    # Duplicate detection
    # ========================================================

    duplicate_columns = [
        col
        for col in [
            "start_frame",
            "end_frame"
        ]
        if col in metadata_df.columns
    ]

    if duplicate_columns:

        duplicates = (
            metadata_df
            .duplicated(
                subset=duplicate_columns
            )
            .sum()
        )

        result[
            "metadata_duplicates"
        ] = int(
            duplicates
        )

        if duplicates > 0:

            result["errors"].append(
                "metadata_duplicates"
            )


    # ========================================================
    # Validate frame ranges
    # ========================================================

    try:

        starts = pd.to_numeric(
            metadata_df[
                "start_frame"
            ],
            errors="coerce"
        )

        ends = pd.to_numeric(
            metadata_df[
                "end_frame"
            ],
            errors="coerce"
        )

        invalid_range = (
            starts.isna()
            |
            ends.isna()
            |
            (ends < starts)
            |
            ((ends - starts + 1) != CLIP_LENGTH)
        )

        invalid_count = int(
            invalid_range.sum()
        )

        if invalid_count > 0:

            result["errors"].append(
                "invalid_clip_frame_range"
            )

            print(
                f"❌ Invalid clip ranges : "
                f"{invalid_count}"
            )

        else:

            print(
                "✅ Clip frame ranges valid"
            )

    except Exception as e:

        result["errors"].append(
            "frame_range_validation_error"
        )

        print(
            f"❌ Frame range validation error: "
            f"{type(e).__name__}: {e}"
        )


    # ========================================================
    # Find shard files
    # ========================================================

    shard_files = sorted(
        paths["shard_dir"].glob(
            "*.npz"
        )
    )

    result[
        "shard_files"
    ] = len(
        shard_files
    )

    print(
        f"Shard files   : "
        f"{len(shard_files)}"
    )


    if not shard_files:

        result["errors"].append(
            "no_shard_files"
        )

        print(
            "❌ No .npz shard files found"
        )

        return result


    # ========================================================
    # Read shards
    #
    # np.load is used one shard at a time.
    # ========================================================

    total_shard_clips = 0

    for shard_path in shard_files:

        try:

            with np.load(
                shard_path,
                allow_pickle=False
            ) as data:

                keys = list(
                    data.files
                )

                if not keys:

                    result[
                        "sample_empty_shards"
                    ] += 1

                    continue


                # ------------------------------------------------
                # Determine clip array
                # ------------------------------------------------

                array = None

                for key in [
                    "clips",
                    "clip",
                    "data",
                    "arr_0"
                ]:

                    if key in data:

                        array = data[
                            key
                        ]

                        break


                if array is None:

                    # fallback:
                    # use first array in shard

                    array = data[
                        keys[0]
                    ]


                # ------------------------------------------------
                # Count clips
                # ------------------------------------------------

                if array.ndim >= 1:

                    total_shard_clips += (
                        array.shape[0]
                    )


                # ------------------------------------------------
                # Sample shape
                # ------------------------------------------------

                if (
                    array.ndim >= 2
                    and array.shape[0] > 0
                ):

                    sample = array[0]

                    # Expected:
                    # first dimension of sample = CLIP_LENGTH

                    if (
                        sample.ndim >= 1
                        and sample.shape[0]
                        != CLIP_LENGTH
                    ):

                        result[
                            "sample_shape_errors"
                        ] += 1

                    # dtype check

                    if sample.dtype != np.uint8:

                        result[
                            "sample_dtype_errors"
                        ] += 1


        except Exception:

            result[
                "shard_read_errors"
            ] += 1


    result[
        "shard_clips"
    ] = total_shard_clips


    print(
        f"Shard clips   : "
        f"{total_shard_clips}"
    )


    # ========================================================
    # Compare metadata ↔ shards
    # ========================================================

    if (
        total_shard_clips
        != len(metadata_df)
    ):

        result["errors"].append(
            "metadata_shard_count_mismatch"
        )

        print(
            "❌ Metadata ↔ shard count mismatch"
        )

    else:

        print(
            "✅ Metadata ↔ shard count matched"
        )


    # ========================================================
    # Sample validation errors
    # ========================================================

    if (
        result["shard_read_errors"]
        > 0
    ):

        result["errors"].append(
            "shard_read_error"
        )


    if (
        result["sample_shape_errors"]
        > 0
    ):

        result["errors"].append(
            "sample_shape_error"
        )


    if (
        result["sample_dtype_errors"]
        > 0
    ):

        result["errors"].append(
            "sample_dtype_error"
        )


    if (
        result["sample_empty_shards"]
        > 0
    ):

        result["errors"].append(
            "empty_shard"
        )


    # ========================================================
    # Final status
    # ========================================================

    if not result["errors"]:

        result[
            "valid"
        ] = True

        print()
        print(
            "✅ PHASE 6 VALIDATION PASSED"
        )

    else:

        result[
            "valid"
        ] = False

        print()
        print(
            "❌ PHASE 6 VALIDATION FAILED"
        )

        print(
            "Errors:"
        )

        for error in result[
            "errors"
        ]:

            print(
                f"  ❌ {error}"
            )


    return result

In [16]:
# ============================================================
# PHASE 6 — CELL 15
# Final Validation Batch
# ============================================================

PHASE6_VALIDATION_RESULTS = []

print()
print("=" * 70)
print("PHASE 6 — FINAL VALIDATION")
print("=" * 70)

for position, video_id in enumerate(
    VIDEO_IDS,
    start=1
):

    print()
    print(
        f"[{position}/{len(VIDEO_IDS)}] "
        f"{video_id}"
    )

    try:

        result = validate_phase6_video(
            video_id
        )

        PHASE6_VALIDATION_RESULTS.append(
            result
        )

    except Exception as e:

        print()
        print(
            f"❌ VALIDATION ERROR: "
            f"{video_id}"
        )

        traceback.print_exc()

        PHASE6_VALIDATION_RESULTS.append({

            "video_id":
                video_id,

            "metadata_exists":
                False,

            "shard_dir_exists":
                False,

            "metadata_rows":
                0,

            "shard_files":
                0,

            "shard_clips":
                0,

            "metadata_duplicates":
                0,

            "metadata_missing_columns":
                0,

            "shard_read_errors":
                0,

            "sample_shape_errors":
                0,

            "sample_dtype_errors":
                0,

            "sample_empty_shards":
                0,

            "valid":
                False,

            "errors": [
                (
                    f"{type(e).__name__}: "
                    f"{e}"
                )
            ]
        })


# ============================================================
# DataFrame
# ============================================================

PHASE6_VALIDATION_DF = pd.DataFrame(
    PHASE6_VALIDATION_RESULTS
)


# ============================================================
# Display summary
# ============================================================

print()
print("=" * 70)
print("PHASE 6 — VALIDATION SUMMARY")
print("=" * 70)

if not PHASE6_VALIDATION_DF.empty:

    display_columns = [
        "video_id",
        "metadata_rows",
        "shard_files",
        "shard_clips",
        "metadata_duplicates",
        "metadata_missing_columns",
        "shard_read_errors",
        "sample_shape_errors",
        "sample_dtype_errors",
        "valid",
    ]

    print(
        PHASE6_VALIDATION_DF[
            display_columns
        ].to_string(
            index=False
        )
    )


# ============================================================
# Final
# ============================================================

all_valid = (
    not PHASE6_VALIDATION_DF.empty
    and
    PHASE6_VALIDATION_DF[
        "valid"
    ].all()
)


print()

if all_valid:

    print("=" * 70)
    print(
        "✅ PHASE 6 VALIDATION PASSED"
    )
    print(
        "All videos are ready for PHASE 7."
    )
    print("=" * 70)

else:

    print("=" * 70)
    print(
        "⚠️ PHASE 6 VALIDATION REQUIRES ATTENTION"
    )
    print(
        "Do NOT start PHASE 7 yet."
    )
    print("=" * 70)


PHASE 6 — FINAL VALIDATION

[1/3] video001

PHASE 6 VALIDATION : video001
Metadata : C:\LipReadingSSL\output\video001\clip_metadata.csv
Shards   : C:\LipReadingSSL\output\video001\clip_npy


C:\Users\User\AppData\Local\Temp\ipykernel_32496\677595650.py:121: DtypeWarning: Columns (0,1,2,3,9,10,13,16) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv(


Metadata rows : 83069
❌ Invalid clip ranges : 42993
Shard files   : 79
Shard clips   : 40076
❌ Metadata ↔ shard count mismatch

❌ PHASE 6 VALIDATION FAILED
Errors:
  ❌ invalid_clip_frame_range
  ❌ metadata_shard_count_mismatch

[2/3] video002

PHASE 6 VALIDATION : video002
Metadata : C:\LipReadingSSL\output\video002\clip_metadata.csv
Shards   : C:\LipReadingSSL\output\video002\clip_npy


C:\Users\User\AppData\Local\Temp\ipykernel_32496\677595650.py:121: DtypeWarning: Columns (10,13,16) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv(


Metadata rows : 70522
❌ Invalid clip ranges : 38860
Shard files   : 62
Shard clips   : 31662
❌ Metadata ↔ shard count mismatch

❌ PHASE 6 VALIDATION FAILED
Errors:
  ❌ invalid_clip_frame_range
  ❌ metadata_shard_count_mismatch

[3/3] video003

PHASE 6 VALIDATION : video003
Metadata : C:\LipReadingSSL\output\video003\clip_metadata.csv
Shards   : C:\LipReadingSSL\output\video003\clip_npy
Metadata rows : 26273
✅ Clip frame ranges valid
Shard files   : 52
Shard clips   : 26273
✅ Metadata ↔ shard count matched

✅ PHASE 6 VALIDATION PASSED

PHASE 6 — VALIDATION SUMMARY
video_id  metadata_rows  shard_files  shard_clips  metadata_duplicates  metadata_missing_columns  shard_read_errors  sample_shape_errors  sample_dtype_errors  valid
video001          83069           79        40076                    0                         0                  0                    0                    0  False
video002          70522           62        31662                    0                         0    

In [17]:
# ============================================================
# PHASE 6 — CELL 16
# Inspect Existing Shards
#
# IMPORTANT:
# - READ ONLY
# - NEVER modifies clip_npy
# - NEVER modifies clip_metadata.csv
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 70)
print("PHASE 6 — SHARD INSPECTION")
print("=" * 70)

REPAIR_VIDEO_IDS = [
    "video001",
    "video002",
]

for video_id in REPAIR_VIDEO_IDS:

    print()
    print("=" * 70)
    print(f"INSPECTING : {video_id}")
    print("=" * 70)

    output_dir = (
        PROJECT_ROOT
        / "output"
        / video_id
    )

    shard_dir = (
        output_dir
        / "clip_npy"
    )

    metadata_path = (
        output_dir
        / "clip_metadata.csv"
    )

    print(f"Shard dir : {shard_dir}")
    print(f"Metadata  : {metadata_path}")

    if not shard_dir.exists():
        print("❌ Shard directory not found")
        continue

    shard_files = sorted(
        shard_dir.glob("*.npz")
    )

    print()
    print(f"Shard files : {len(shard_files)}")

    if not shard_files:
        print("❌ No .npz shard files found")
        continue

    # --------------------------------------------------------
    # Inspect first shard only
    # --------------------------------------------------------

    first_shard = shard_files[0]

    print()
    print(f"First shard : {first_shard.name}")

    try:

        with np.load(
            first_shard,
            allow_pickle=False
        ) as data:

            print()
            print("Keys inside shard:")

            for key in data.files:

                arr = data[key]

                print(
                    f"  {key:25s}"
                    f" shape={arr.shape}"
                    f" dtype={arr.dtype}"
                )

    except Exception as e:

        print(
            f"❌ Cannot inspect shard: "
            f"{type(e).__name__}: {e}"
        )

    # --------------------------------------------------------
    # Current metadata
    # --------------------------------------------------------

    if metadata_path.exists():

        try:

            metadata_df = pd.read_csv(
                metadata_path,
                low_memory=False
            )

            print()
            print(
                f"Current metadata rows : "
                f"{len(metadata_df)}"
            )

            print()
            print("Metadata columns:")

            for col in metadata_df.columns:
                print(f"  - {col}")

            print()
            print("First 3 metadata rows:")

            print(
                metadata_df
                .head(3)
                .to_string(index=False)
            )

        except Exception as e:

            print(
                f"❌ Cannot read metadata: "
                f"{type(e).__name__}: {e}"
            )

print()
print("=" * 70)
print("CELL 16 COMPLETE — READ ONLY")
print("=" * 70)

PHASE 6 — SHARD INSPECTION

INSPECTING : video001
Shard dir : C:\LipReadingSSL\output\video001\clip_npy
Metadata  : C:\LipReadingSSL\output\video001\clip_metadata.csv

Shard files : 79

First shard : shard_000000.npz

Keys inside shard:
  clips                     shape=(512, 16, 96, 96) dtype=uint8
  start_frames              shape=(512,) dtype=int32
  end_frames                shape=(512,) dtype=int32
  clip_indices              shape=(512,) dtype=int32

Current metadata rows : 83069

Metadata columns:
  - clip_id
  - video
  - jpg_folder
  - npy_path
  - start_frame
  - end_frame
  - num_frames
  - clip_length
  - stride
  - color_mode
  - video_id
  - clip_index
  - shard_id
  - dtype
  - height
  - width
  - storage

First 3 metadata rows:
clip_id video jpg_folder npy_path  start_frame  end_frame  num_frames  clip_length  stride color_mode video_id  clip_index  shard_id                 dtype  height  width   storage
    NaN   NaN        NaN      NaN            0         15        

In [24]:
# ============================================================
# PHASE 6 — CELL 17
# REPAIR METADATA v2
#
# SOURCE OF TRUTH:
#   Existing .npz files inside clip_npy
#
# IMPORTANT:
#   - READS EVERY .npz SHARD
#   - DOES NOT MODIFY clip_npy
#   - DOES NOT REGENERATE CLIPS
#   - DOES NOT DELETE ANY FILE
#   - BACKUPS OLD clip_metadata.csv
#   - WRITES ONE METADATA ROW PER ACTUAL CLIP
# ============================================================

from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import shutil


print("=" * 70)
print("PHASE 6 — REPAIR METADATA v2")
print("=" * 70)


REPAIR_VIDEO_IDS = [
    "video001",
    "video002",
]


# ============================================================
# Helper
# ============================================================

def get_video_output_dir(video_id):

    return (
        PROJECT_ROOT
        / "output"
        / video_id
    )


# ============================================================
# Repair one video
# ============================================================

def repair_metadata_v2(video_id):

    print()
    print("=" * 70)
    print(f"REPAIR v2 : {video_id}")
    print("=" * 70)

    output_dir = get_video_output_dir(
        video_id
    )

    shard_dir = (
        output_dir
        / "clip_npy"
    )

    metadata_path = (
        output_dir
        / "clip_metadata.csv"
    )

    backup_dir = (
        output_dir
        / "backups"
    )

    # --------------------------------------------------------
    # Create backup directory
    # --------------------------------------------------------

    backup_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Validate shard directory
    # --------------------------------------------------------

    print(
        f"Output    : {output_dir}"
    )

    print(
        f"Shard dir : {shard_dir}"
    )

    print(
        f"Metadata  : {metadata_path}"
    )

    if not shard_dir.exists():

        raise FileNotFoundError(
            f"clip_npy not found:\n"
            f"{shard_dir}"
        )

    # --------------------------------------------------------
    # IMPORTANT:
    # Read EVERY .npz
    # --------------------------------------------------------

    shard_files = sorted(
        shard_dir.glob("*.npz")
    )

    print()
    print(
        f"NPZ shards found : "
        f"{len(shard_files)}"
    )

    if not shard_files:

        raise RuntimeError(
            "No .npz files found."
        )

    # --------------------------------------------------------
    # Backup current metadata
    # --------------------------------------------------------

    backup_path = None

    if metadata_path.exists():

        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        backup_path = (
            backup_dir
            /
            f"clip_metadata_before_repair_v2_{timestamp}.csv"
        )

        shutil.copy2(
            metadata_path,
            backup_path
        )

        print(
            f"Backup created:"
        )

        print(
            f"  {backup_path}"
        )

    # --------------------------------------------------------
    # Storage for ALL shard records
    # --------------------------------------------------------

    all_records = []

    all_clip_ids = []

    total_clips_from_shards = 0

    invalid_shards = []

    # --------------------------------------------------------
    # Read EVERY shard
    # --------------------------------------------------------

    for shard_number, shard_path in enumerate(
        shard_files,
        start=1
    ):

        try:

            with np.load(
                shard_path,
                allow_pickle=False
            ) as data:

                # --------------------------------------------
                # Required keys
                # --------------------------------------------

                required_keys = [

                    "clips",
                    "start_frames",
                    "end_frames",
                    "clip_indices",

                ]

                missing_keys = [

                    key
                    for key in required_keys
                    if key not in data.files

                ]

                if missing_keys:

                    raise RuntimeError(
                        f"Missing keys: "
                        f"{missing_keys}"
                    )

                # --------------------------------------------
                # Load arrays
                # --------------------------------------------

                clips = data[
                    "clips"
                ]

                start_frames = data[
                    "start_frames"
                ]

                end_frames = data[
                    "end_frames"
                ]

                clip_indices = data[
                    "clip_indices"
                ]

                # --------------------------------------------
                # Number of clips
                # --------------------------------------------

                n_clips = len(
                    clip_indices
                )

                # --------------------------------------------
                # Internal consistency
                # --------------------------------------------

                if len(clips) != n_clips:

                    raise RuntimeError(
                        "clips count mismatch"
                    )

                if len(start_frames) != n_clips:

                    raise RuntimeError(
                        "start_frames count mismatch"
                    )

                if len(end_frames) != n_clips:

                    raise RuntimeError(
                        "end_frames count mismatch"
                    )

                # --------------------------------------------
                # Process EVERY clip in this shard
                # --------------------------------------------

                for local_index in range(
                    n_clips
                ):

                    clip_id = int(
                        clip_indices[
                            local_index
                        ]
                    )

                    start_frame = int(
                        start_frames[
                            local_index
                        ]
                    )

                    end_frame = int(
                        end_frames[
                            local_index
                        ]
                    )

                    clip_array = clips[
                        local_index
                    ]

                    # ----------------------------------------
                    # Basic frame validation
                    # ----------------------------------------

                    if (
                        start_frame < 0
                        or end_frame < start_frame
                    ):

                        print(
                            f"⚠️ Invalid range "
                            f"in {shard_path.name} "
                            f"clip {clip_id}"
                        )

                        continue

                    # ----------------------------------------
                    # Record
                    # ----------------------------------------

                    all_records.append({

                        "clip_id":
                            clip_id,

                        "video":
                            video_id,

                        "start_frame":
                            start_frame,

                        "end_frame":
                            end_frame,

                        "num_frames":
                            end_frame
                            - start_frame
                            + 1,

                        "shard":
                            shard_path.name,

                        "shard_index":
                            local_index,

                        "clip_shape":
                            str(
                                tuple(
                                    clip_array.shape
                                )
                            ),

                        "dtype":
                            str(
                                clip_array.dtype
                            ),

                    })

                    all_clip_ids.append(
                        clip_id
                    )

                # --------------------------------------------
                # Count ALL clips
                # --------------------------------------------

                total_clips_from_shards += (
                    n_clips
                )

        except Exception as e:

            invalid_shards.append({

                "shard":
                    shard_path.name,

                "error":
                    f"{type(e).__name__}: {e}",

            })

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        print(
    f"[{shard_number:>3}/"
    f"{len(shard_files)}] "
    f"{shard_path.name:<20} "
    f"clips={len(clip_indices)}"
)

    # ========================================================
    # IMPORTANT:
    # Stop if any shard failed
    # ========================================================

    if invalid_shards:

        print()
        print(
            "❌ Some shards could not be read."
        )

        for item in invalid_shards:

            print(
                f"  {item['shard']}: "
                f"{item['error']}"
            )

        raise RuntimeError(
            "Repair aborted because "
            "one or more shards are invalid."
        )

    # ========================================================
    # Build DataFrame
    # ========================================================

    repaired_df = pd.DataFrame(
        all_records
    )

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if repaired_df.empty:

        raise RuntimeError(
            "No metadata records generated."
        )

    # ========================================================
    # Duplicate detection
    # ========================================================

    duplicate_mask = (
        repaired_df[
            "clip_id"
        ].duplicated(
            keep=False
        )
    )

    duplicate_count = int(
        duplicate_mask.sum()
    )

    duplicate_unique_ids = int(
        repaired_df.loc[
            duplicate_mask,
            "clip_id"
        ].nunique()
    )

    print()
    print("=" * 70)
    print("REPAIR v2 RESULT")
    print("=" * 70)

    print(
        f"Shard files          : "
        f"{len(shard_files)}"
    )

    print(
        f"Shard clips          : "
        f"{total_clips_from_shards}"
    )

    print(
        f"Metadata records     : "
        f"{len(repaired_df)}"
    )

    print(
        f"Duplicate clip IDs   : "
        f"{duplicate_unique_ids}"
    )

    # ========================================================
    # CRITICAL SAFETY CHECK
    #
    # Expected:
    #
    #   shard clips == metadata rows
    #
    # If not:
    #   DO NOT overwrite metadata
    # ========================================================

    if (
        len(repaired_df)
        !=
        total_clips_from_shards
    ):

        raise RuntimeError(
            "SAFETY CHECK FAILED: "
            "metadata record count does not "
            "match total clips across ALL shards."
        )

    # ========================================================
    # Remove duplicate IDs
    # ========================================================

    if duplicate_count > 0:

        print()
        print(
            "⚠️ Duplicate clip IDs detected."
        )

        print(
            "Keeping first occurrence."
        )

        repaired_df = (
            repaired_df
            .drop_duplicates(
                subset=[
                    "clip_id"
                ],
                keep="first"
            )
            .reset_index(
                drop=True
            )
        )

    # ========================================================
    # Add repair timestamp
    # ========================================================

    repaired_df[
        "repaired_at"
    ] = datetime.utcnow().isoformat() + "Z"

    # ========================================================
    # FINAL SAFETY CHECK
    # ========================================================

    final_metadata_count = len(
        repaired_df
    )

    final_unique_ids = (
        repaired_df[
            "clip_id"
        ]
        .nunique()
    )

    print(
        f"Final metadata rows : "
        f"{final_metadata_count}"
    )

    print(
        f"Unique clip IDs     : "
        f"{final_unique_ids}"
    )

    # --------------------------------------------------------
    # If duplicates existed, count can differ
    # --------------------------------------------------------

    if duplicate_unique_ids > 0:

        raise RuntimeError(
            "Duplicate clip IDs exist across shards. "
            "Metadata was NOT overwritten."
        )

    # ========================================================
    # Sort by clip ID
    # ========================================================

    repaired_df = (
        repaired_df
        .sort_values(
            "clip_id"
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # WRITE
    # ========================================================

    repaired_df.to_csv(
        metadata_path,
        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # Final report
    # ========================================================

    print()
    print(
        f"✅ Metadata repaired successfully."
    )

    print(
        f"   Rows : {len(repaired_df)}"
    )

    print(
        f"   File : {metadata_path}"
    )

    print()
    print(
        "clip_npy status:"
    )

    print(
        "   ✅ NOT MODIFIED"
    )

    return {

        "video_id":
            video_id,

        "shard_files":
            len(shard_files),

        "shard_clips":
            total_clips_from_shards,

        "metadata_rows":
            len(repaired_df),

        "unique_clip_ids":
            final_unique_ids,

        "duplicates":
            duplicate_unique_ids,

        "metadata_path":
            str(metadata_path),

        "backup_path":
            (
                str(backup_path)
                if backup_path
                else None
            ),

        "status":
            "REPAIRED",

    }


# ============================================================
# RUN REPAIR
# ============================================================

PHASE6_REPAIR_V2_RESULTS = []


for video_id in REPAIR_VIDEO_IDS:

    try:

        result = repair_metadata_v2(
            video_id
        )

        PHASE6_REPAIR_V2_RESULTS.append(
            result
        )

    except Exception as e:

        print()
        print("=" * 70)
        print(
            f"❌ REPAIR v2 FAILED : "
            f"{video_id}"
        )
        print("=" * 70)

        print(
            f"{type(e).__name__}: {e}"
        )

        PHASE6_REPAIR_V2_RESULTS.append({

            "video_id":
                video_id,

            "status":
                "FAILED",

            "error":
                f"{type(e).__name__}: {e}",

        })


# ============================================================
# Summary
# ============================================================

print()
print("=" * 70)
print("PHASE 6 — REPAIR v2 SUMMARY")
print("=" * 70)

repair_v2_df = pd.DataFrame(
    PHASE6_REPAIR_V2_RESULTS
)

if not repair_v2_df.empty:

    print(
        repair_v2_df.to_string(
            index=False
        )
    )

print("=" * 70)

PHASE 6 — REPAIR METADATA v2

REPAIR v2 : video001
Output    : C:\LipReadingSSL\output\video001
Shard dir : C:\LipReadingSSL\output\video001\clip_npy
Metadata  : C:\LipReadingSSL\output\video001\clip_metadata.csv

NPZ shards found : 79
Backup created:
  C:\LipReadingSSL\output\video001\backups\clip_metadata_before_repair_v2_20260829_020822.csv
[  1/79] shard_000000.npz     clips=512
[  2/79] shard_000001.npz     clips=512
[  3/79] shard_000002.npz     clips=512
[  4/79] shard_000003.npz     clips=512
[  5/79] shard_000004.npz     clips=512
[  6/79] shard_000005.npz     clips=512
[  7/79] shard_000006.npz     clips=512
[  8/79] shard_000007.npz     clips=512
[  9/79] shard_000008.npz     clips=512
[ 10/79] shard_000009.npz     clips=512
[ 11/79] shard_000010.npz     clips=512
[ 12/79] shard_000011.npz     clips=512
[ 13/79] shard_000012.npz     clips=512
[ 14/79] shard_000013.npz     clips=512
[ 15/79] shard_000014.npz     clips=512
[ 16/79] shard_000015.npz     clips=512
[ 17/79] shard

In [26]:
# ============================================================
# PHASE 6 — CELL 17A
# READ-ONLY DUPLICATE DIAGNOSIS
#
# IMPORTANT:
#   - READ ONLY
#   - DOES NOT MODIFY clip_npy
#   - DOES NOT MODIFY clip_metadata.csv
#   - DOES NOT DELETE ANY FILE
#   - READS EVERY .npz SHARD
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 70)
print("PHASE 6 — READ-ONLY DUPLICATE DIAGNOSIS")
print("=" * 70)


DIAGNOSIS_VIDEO_IDS = [
    "video001",
    "video002",
]


# ============================================================
# Diagnose one video
# ============================================================

def diagnose_shard_indices(video_id):

    print()
    print("=" * 70)
    print(f"DIAGNOSIS : {video_id}")
    print("=" * 70)

    output_dir = (
        PROJECT_ROOT
        / "output"
        / video_id
    )

    shard_dir = (
        output_dir
        / "clip_npy"
    )

    if not shard_dir.exists():

        print(
            f"❌ Shard directory not found:"
        )

        print(
            f"   {shard_dir}"
        )

        return None

    shard_files = sorted(
        shard_dir.glob("*.npz")
    )

    print(
        f"Shard directory : {shard_dir}"
    )

    print(
        f"Shard files     : {len(shard_files)}"
    )

    if not shard_files:

        print(
            "❌ No .npz shards found."
        )

        return None

    # ========================================================
    # Global diagnostic storage
    # ========================================================

    all_clip_ids = []

    shard_reports = []

    first_examples = []

    # ========================================================
    # Read EVERY shard
    # ========================================================

    for shard_number, shard_path in enumerate(
        shard_files,
        start=1
    ):

        try:

            with np.load(
                shard_path,
                allow_pickle=False
            ) as data:

                # ------------------------------------------------
                # Check keys
                # ------------------------------------------------

                if "clip_indices" not in data.files:

                    print(
                        f"[{shard_number:>3}/"
                        f"{len(shard_files)}] "
                        f"{shard_path.name:<20} "
                        f"❌ missing clip_indices"
                    )

                    shard_reports.append({

                        "shard":
                            shard_path.name,

                        "clips":
                            0,

                        "min_id":
                            None,

                        "max_id":
                            None,

                        "unique_ids":
                            0,

                        "sequential":
                            False,

                        "error":
                            "missing_clip_indices",

                    })

                    continue

                clip_indices = np.asarray(
                    data[
                        "clip_indices"
                    ]
                )

                n_clips = len(
                    clip_indices
                )

                # ------------------------------------------------
                # Basic values
                # ------------------------------------------------

                if n_clips > 0:

                    min_id = int(
                        clip_indices.min()
                    )

                    max_id = int(
                        clip_indices.max()
                    )

                    unique_ids = int(
                        np.unique(
                            clip_indices
                        ).size
                    )

                else:

                    min_id = None
                    max_id = None
                    unique_ids = 0

                # ------------------------------------------------
                # Check whether IDs are local 0..N-1
                # ------------------------------------------------

                expected_local = np.arange(
                    n_clips,
                    dtype=clip_indices.dtype
                )

                is_local_sequence = (
                    n_clips > 0
                    and
                    np.array_equal(
                        clip_indices,
                        expected_local
                    )
                )

                # ------------------------------------------------
                # Check whether IDs are sequential
                # ------------------------------------------------

                if n_clips > 1:

                    differences = np.diff(
                        clip_indices
                    )

                    is_sequential = bool(
                        np.all(
                            differences == 1
                        )
                    )

                else:

                    is_sequential = True

                # ------------------------------------------------
                # Store IDs
                # ------------------------------------------------

                all_clip_ids.extend(
                    clip_indices.astype(
                        np.int64
                    ).tolist()
                )

                # ------------------------------------------------
                # Save first few IDs
                # ------------------------------------------------

                preview_ids = (
                    clip_indices[
                        :10
                    ].astype(
                        np.int64
                    ).tolist()
                )

                last_ids = (
                    clip_indices[
                        -10:
                    ].astype(
                        np.int64
                    ).tolist()
                )

                first_examples.append({

                    "shard":
                        shard_path.name,

                    "first_10":
                        preview_ids,

                    "last_10":
                        last_ids,

                })

                # ------------------------------------------------
                # Report
                # ------------------------------------------------

                shard_reports.append({

                    "shard":
                        shard_path.name,

                    "clips":
                        n_clips,

                    "min_id":
                        min_id,

                    "max_id":
                        max_id,

                    "unique_ids":
                        unique_ids,

                    "sequential":
                        is_sequential,

                    "local_0_to_n_minus_1":
                        is_local_sequence,

                    "error":
                        None,

                })

                print(
                    f"[{shard_number:>3}/"
                    f"{len(shard_files)}] "
                    f"{shard_path.name:<20} "
                    f"clips={n_clips:<4} "
                    f"range={min_id}..{max_id} "
                    f"unique={unique_ids:<4} "
                    f"local={is_local_sequence}"
                )

        except Exception as e:

            print(
                f"[{shard_number:>3}/"
                f"{len(shard_files)}] "
                f"{shard_path.name:<20} "
                f"❌ "
                f"{type(e).__name__}: {e}"
            )

            shard_reports.append({

                "shard":
                    shard_path.name,

                "clips":
                    0,

                "min_id":
                    None,

                "max_id":
                    None,

                "unique_ids":
                    0,

                "sequential":
                    False,

                "local_0_to_n_minus_1":
                    False,

                "error":
                    f"{type(e).__name__}: {e}",

            })

    # ========================================================
    # Global analysis
    # ========================================================

    clip_id_array = np.asarray(
        all_clip_ids,
        dtype=np.int64
    )

    total_clips = int(
        len(clip_id_array)
    )

    unique_global = int(
        np.unique(
            clip_id_array
        ).size
    )

    duplicate_count = (
        total_clips
        -
        unique_global
    )

    # ========================================================
    # Duplicate IDs
    # ========================================================

    if total_clips > 0:

        values, counts = np.unique(
            clip_id_array,
            return_counts=True
        )

        duplicate_ids = values[
            counts > 1
        ]

        duplicate_occurrences = counts[
            counts > 1
        ]

    else:

        duplicate_ids = np.array(
            [],
            dtype=np.int64
        )

        duplicate_occurrences = np.array(
            [],
            dtype=np.int64
        )

    # ========================================================
    # Determine pattern
    # ========================================================

    local_flags = [

        report.get(
            "local_0_to_n_minus_1",
            False
        )

        for report in shard_reports

        if report.get(
            "error"
        ) is None

    ]

    all_shards_local = (
        len(local_flags) == len(shard_files)
        and
        all(local_flags)
    )

    # ========================================================
    # Summary
    # ========================================================

    print()
    print("=" * 70)
    print(f"GLOBAL RESULT : {video_id}")
    print("=" * 70)

    print(
        f"Total clips across shards : "
        f"{total_clips}"
    )

    print(
        f"Unique clip IDs            : "
        f"{unique_global}"
    )

    print(
        f"Duplicate occurrences      : "
        f"{duplicate_count}"
    )

    print(
        f"Duplicate unique IDs       : "
        f"{len(duplicate_ids)}"
    )

    print(
        f"All shards use local IDs   : "
        f"{all_shards_local}"
    )

    # ========================================================
    # Duplicate examples
    # ========================================================

    if len(duplicate_ids) > 0:

        print()
        print(
            "FIRST DUPLICATE CLIP IDs"
        )

        limit = min(
            20,
            len(duplicate_ids)
        )

        for i in range(limit):

            print(
                f"  clip_id="
                f"{int(duplicate_ids[i])}"
                f"  occurrences="
                f"{int(duplicate_occurrences[i])}"
            )

    else:

        print()
        print(
            "✅ No duplicate clip IDs."
        )

    # ========================================================
    # Shard pattern table
    # ========================================================

    report_df = pd.DataFrame(
        shard_reports
    )

    print()
    print(
        "=" * 70
    )
    print(
        "SHARD PATTERN"
    )
    print(
        "=" * 70
    )

    if not report_df.empty:

        print(
            report_df[
                [
                    "shard",
                    "clips",
                    "min_id",
                    "max_id",
                    "unique_ids",
                    "sequential",
                    "local_0_to_n_minus_1",
                ]
            ].to_string(
                index=False
            )
        )

    # ========================================================
    # First / last IDs
    # ========================================================

    print()
    print(
        "=" * 70
    )
    print(
        "SHARD ID EXAMPLES"
    )
    print(
        "=" * 70
    )

    for example in first_examples:

        print()
        print(
            example["shard"]
        )

        print(
            f"  First 10 : "
            f"{example['first_10']}"
        )

        print(
            f"  Last 10  : "
            f"{example['last_10']}"
        )

    # ========================================================
    # Interpretation
    # ========================================================

    print()
    print(
        "=" * 70
    )
    print(
        "DIAGNOSIS"
    )
    print(
        "=" * 70
    )

    if all_shards_local:

        print(
            "⚠️ PATTERN DETECTED:"
        )

        print(
            "Every shard appears to use "
            "local clip indices."
        )

        print()
        print(
            "Example pattern:"
        )

        print(
            "  shard_000000 → 0..511"
        )

        print(
            "  shard_000001 → 0..511"
        )

        print(
            "  shard_000002 → 0..511"
        )

        print()
        print(
            "This means clip_indices "
            "cannot be used directly "
            "as global clip IDs."
        )

        print()
        print(
            "A global ID should instead "
            "be derived from shard order "
            "and shard-local index."
        )

    elif duplicate_count > 0:

        print(
            "⚠️ Duplicate IDs detected, "
            "but the pattern is NOT simply "
            "local 0..N-1 IDs."
        )

        print()
        print(
            "Do NOT repair automatically yet."
        )

        print(
            "Further investigation is required."
        )

    else:

        print(
            "✅ No duplicate ID pattern detected."
        )

    # ========================================================
    # Return diagnostics
    # ========================================================

    return {

        "video_id":
            video_id,

        "shard_files":
            len(shard_files),

        "total_clips":
            total_clips,

        "unique_clip_ids":
            unique_global,

        "duplicate_occurrences":
            duplicate_count,

        "duplicate_unique_ids":
            len(duplicate_ids),

        "all_shards_local":
            all_shards_local,

        "shard_reports":
            report_df,

        "duplicate_ids":
            duplicate_ids,

        "duplicate_occurrences":
            duplicate_occurrences,

    }


# ============================================================
# RUN DIAGNOSIS
# ============================================================

PHASE6_DIAGNOSIS = {}


for video_id in DIAGNOSIS_VIDEO_IDS:

    try:

        PHASE6_DIAGNOSIS[
            video_id
        ] = diagnose_shard_indices(
            video_id
        )

    except Exception as e:

        print()
        print(
            "=" * 70
        )

        print(
            f"❌ DIAGNOSIS FAILED : "
            f"{video_id}"
        )

        print(
            f"{type(e).__name__}: {e}"
        )

        PHASE6_DIAGNOSIS[
            video_id
        ] = None


# ============================================================
# Final summary
# ============================================================

print()
print("=" * 70)
print("PHASE 6 — DUPLICATE DIAGNOSIS SUMMARY")
print("=" * 70)

for video_id, result in PHASE6_DIAGNOSIS.items():

    if result is None:

        print(
            f"{video_id}: FAILED"
        )

    else:

        print(
            f"{video_id}: "
            f"shards={result['shard_files']} | "
            f"clips={result['total_clips']} | "
            f"unique={result['unique_clip_ids']} | "
            f"duplicates={result['duplicate_occurrences']} | "
            f"all_local={result['all_shards_local']}"
        )

print("=" * 70)
print("CELL 17A COMPLETE — READ ONLY")
print("=" * 70)

PHASE 6 — READ-ONLY DUPLICATE DIAGNOSIS

DIAGNOSIS : video001
Shard directory : C:\LipReadingSSL\output\video001\clip_npy
Shard files     : 79
[  1/79] shard_000000.npz     clips=512  range=0..511 unique=512  local=True
[  2/79] shard_000001.npz     clips=512  range=0..511 unique=512  local=True
[  3/79] shard_000002.npz     clips=512  range=0..511 unique=512  local=True
[  4/79] shard_000003.npz     clips=512  range=0..511 unique=512  local=True
[  5/79] shard_000004.npz     clips=512  range=0..511 unique=512  local=True
[  6/79] shard_000005.npz     clips=512  range=0..511 unique=512  local=True
[  7/79] shard_000006.npz     clips=512  range=0..511 unique=512  local=True
[  8/79] shard_000007.npz     clips=512  range=0..511 unique=512  local=True
[  9/79] shard_000008.npz     clips=512  range=0..511 unique=512  local=True
[ 10/79] shard_000009.npz     clips=512  range=0..511 unique=512  local=True
[ 11/79] shard_000010.npz     clips=512  range=0..511 unique=512  local=True
[ 12/79] s

In [29]:
# ============================================================
# PHASE 6 — CELL 17B
# REPAIR METADATA v3 — SAFE VERSION
#
# PURPOSE:
#   Rebuild clip_metadata.csv from EVERY .npz shard.
#
# IMPORTANT:
#   - READ ONLY on clip_npy
#   - DOES NOT MODIFY any .npz
#   - DOES NOT regenerate clips
#   - DOES NOT DELETE old metadata
#   - BACKUP uses COPY, not MOVE
#   - Existing metadata remains untouched until ALL checks pass
#   - Uses GLOBAL clip IDs from shard order
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import numpy as np
import pandas as pd
import traceback


print("=" * 70)
print("PHASE 6 — REPAIR METADATA v3 SAFE")
print("=" * 70)


# ============================================================
# Videos requiring repair
# ============================================================

REPAIR_VIDEO_IDS = [
    "video001",
    "video002",
]


# ============================================================
# Repair timestamp
# ============================================================

REPAIR_TIMESTAMP = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


# ============================================================
# Repair function
# ============================================================

def repair_metadata_v3_safe(video_id):

    print()
    print("=" * 70)
    print(f"REPAIR v3 SAFE : {video_id}")
    print("=" * 70)

    # ========================================================
    # Paths
    # ========================================================

    output_dir = (
        PROJECT_ROOT
        / "output"
        / video_id
    )

    shard_dir = (
        output_dir
        / "clip_npy"
    )

    metadata_path = (
        output_dir
        / "clip_metadata.csv"
    )

    backup_dir = (
        output_dir
        / "backups"
        / "clip_metadata_before_repair_v3"
    )

    temp_metadata_path = (
        output_dir
        / (
            "clip_metadata_repair_v3_"
            f"{REPAIR_TIMESTAMP}.tmp.csv"
        )
    )

    # ========================================================
    # Print paths
    # ========================================================

    print(
        f"Output    : {output_dir}"
    )

    print(
        f"Shard dir : {shard_dir}"
    )

    print(
        f"Metadata  : {metadata_path}"
    )

    # ========================================================
    # Validate paths
    # ========================================================

    if not output_dir.exists():

        raise FileNotFoundError(
            f"Output directory not found:\n"
            f"{output_dir}"
        )

    if not shard_dir.exists():

        raise FileNotFoundError(
            f"Shard directory not found:\n"
            f"{shard_dir}"
        )

    # ========================================================
    # Find EVERY NPZ shard
    # ========================================================

    shard_files = sorted(
        shard_dir.glob("*.npz")
    )

    print()
    print(
        f"NPZ shards found : "
        f"{len(shard_files)}"
    )

    if not shard_files:

        raise RuntimeError(
            "No .npz shard files found."
        )

    # ========================================================
    # Backup existing metadata SAFELY
    #
    # COPY instead of MOVE
    # ========================================================

    if metadata_path.exists():

        backup_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        backup_path = (
            backup_dir
            /
            (
                "clip_metadata_"
                f"{REPAIR_TIMESTAMP}.csv"
            )
        )

        counter = 1

        while backup_path.exists():

            backup_path = (
                backup_dir
                /
                (
                    "clip_metadata_"
                    f"{REPAIR_TIMESTAMP}_"
                    f"{counter}.csv"
                )
            )

            counter += 1

        shutil.copy2(
            metadata_path,
            backup_path
        )

        print()
        print(
            "Backup created by COPY:"
        )

        print(
            f"  {backup_path}"
        )

    else:

        print()
        print(
            "No existing metadata CSV."
        )

    # ========================================================
    # Metadata records
    # ========================================================

    metadata_records = []

    global_clip_id = 0

    total_shard_clips = 0

    # ========================================================
    # Read EVERY shard
    # ========================================================

    for shard_number, shard_path in enumerate(
        shard_files,
        start=0
    ):

        try:

            with np.load(
                shard_path,
                allow_pickle=False
            ) as data:

                # ------------------------------------------------
                # Required keys
                # ------------------------------------------------

                required_keys = [
                    "clips",
                    "start_frames",
                    "end_frames",
                    "clip_indices",
                ]

                missing_keys = [
                    key
                    for key in required_keys
                    if key not in data.files
                ]

                if missing_keys:

                    raise RuntimeError(
                        f"Missing keys "
                        f"{missing_keys} in "
                        f"{shard_path.name}"
                    )

                # ------------------------------------------------
                # Read arrays
                # ------------------------------------------------

                clips = data[
                    "clips"
                ]

                start_frames = np.asarray(
                    data[
                        "start_frames"
                    ]
                )

                end_frames = np.asarray(
                    data[
                        "end_frames"
                    ]
                )

                clip_indices = np.asarray(
                    data[
                        "clip_indices"
                    ]
                )

                # ------------------------------------------------
                # Number of clips
                # ------------------------------------------------

                n_clips = len(
                    clips
                )

                # ------------------------------------------------
                # Length checks
                # ------------------------------------------------

                if len(start_frames) != n_clips:

                    raise RuntimeError(
                        f"start_frames length "
                        f"{len(start_frames)} != "
                        f"clips length "
                        f"{n_clips}"
                    )

                if len(end_frames) != n_clips:

                    raise RuntimeError(
                        f"end_frames length "
                        f"{len(end_frames)} != "
                        f"clips length "
                        f"{n_clips}"
                    )

                if len(clip_indices) != n_clips:

                    raise RuntimeError(
                        f"clip_indices length "
                        f"{len(clip_indices)} != "
                        f"clips length "
                        f"{n_clips}"
                    )

                # ------------------------------------------------
                # Local index validation
                #
                # Cell 17A confirmed:
                # each shard uses 0..N-1
                # ------------------------------------------------

                expected_local_indices = np.arange(
                    n_clips,
                    dtype=clip_indices.dtype
                )

                if not np.array_equal(
                    clip_indices,
                    expected_local_indices
                ):

                    raise RuntimeError(
                        f"Unexpected local "
                        f"clip_indices pattern "
                        f"in {shard_path.name}"
                    )

                # ------------------------------------------------
                # Frame validation
                # ------------------------------------------------

                invalid_frame_mask = (
                    start_frames
                    >
                    end_frames
                )

                invalid_count = int(
                    np.count_nonzero(
                        invalid_frame_mask
                    )
                )

                if invalid_count > 0:

                    raise RuntimeError(
                        f"{invalid_count} invalid "
                        f"frame ranges in "
                        f"{shard_path.name}"
                    )

                # ------------------------------------------------
                # GLOBAL OFFSET
                #
                # global ID =
                # previous shard clip count
                # + local index
                # ------------------------------------------------

                shard_global_start = (
                    global_clip_id
                )

                shard_global_end = (
                    global_clip_id
                    + n_clips
                    - 1
                )

                # ------------------------------------------------
                # Build metadata
                # ------------------------------------------------

                for local_index in range(
                    n_clips
                ):

                    metadata_records.append({

                        "clip_id":
                            global_clip_id,

                        "video":
                            video_id,

                        "shard":
                            shard_path.name,

                        "shard_index":
                            local_index,

                        "start_frame":
                            int(
                                start_frames[
                                    local_index
                                ]
                            ),

                        "end_frame":
                            int(
                                end_frames[
                                    local_index
                                ]
                            ),

                    })

                    global_clip_id += 1

                total_shard_clips += (
                    n_clips
                )

                # ------------------------------------------------
                # Progress
                # ------------------------------------------------

                print(
                    f"[{shard_number + 1:>3}/"
                    f"{len(shard_files)}] "
                    f"{shard_path.name:<20} "
                    f"clips={n_clips:<4} "
                    f"global="
                    f"{shard_global_start}"
                    f".."
                    f"{shard_global_end}"
                )

        except Exception:

            print()
            print(
                f"❌ FAILED reading "
                f"{shard_path.name}"
            )

            raise

    # ========================================================
    # Build DataFrame
    # ========================================================

    metadata_df = pd.DataFrame(
        metadata_records
    )

    # ========================================================
    # SAFETY CHECK 1
    # ========================================================

    print()
    print("=" * 70)
    print("REPAIR v3 SAFETY CHECKS")
    print("=" * 70)

    print(
        f"Shard clips      : "
        f"{total_shard_clips}"
    )

    print(
        f"Metadata records : "
        f"{len(metadata_df)}"
    )

    if len(metadata_df) != (
        total_shard_clips
    ):

        raise RuntimeError(
            "Metadata count does not "
            "match shard clip count."
        )

    # ========================================================
    # SAFETY CHECK 2
    # ========================================================

    required_columns = [
        "clip_id",
        "video",
        "shard",
        "shard_index",
        "start_frame",
        "end_frame",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in metadata_df.columns
    ]

    if missing_columns:

        raise RuntimeError(
            f"Missing metadata columns: "
            f"{missing_columns}"
        )

    print(
        "Required columns : OK"
    )

    # ========================================================
    # SAFETY CHECK 3
    # Duplicate global IDs
    # ========================================================

    duplicate_global_ids = int(
        metadata_df[
            "clip_id"
        ]
        .duplicated()
        .sum()
    )

    print(
        f"Duplicate IDs    : "
        f"{duplicate_global_ids}"
    )

    if duplicate_global_ids > 0:

        raise RuntimeError(
            "Duplicate GLOBAL clip IDs "
            "detected."
        )

    # ========================================================
    # SAFETY CHECK 4
    # Continuous global IDs
    # ========================================================

    expected_global_ids = np.arange(
        len(metadata_df),
        dtype=np.int64
    )

    actual_global_ids = (
        metadata_df[
            "clip_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    if not np.array_equal(
        actual_global_ids,
        expected_global_ids
    ):

        raise RuntimeError(
            "Global clip IDs are not "
            "continuous from 0."
        )

    print(
        "Global ID sequence : OK"
    )

    # ========================================================
    # SAFETY CHECK 5
    # Video consistency
    # ========================================================

    video_values = set(
        metadata_df[
            "video"
        ]
        .astype(str)
        .unique()
    )

    if video_values != {
        video_id
    }:

        raise RuntimeError(
            f"Video mismatch: "
            f"{video_values}"
        )

    print(
        "Video ID consistency : OK"
    )

    # ========================================================
    # SAFETY CHECK 6
    # Shard uniqueness
    # ========================================================

    shard_counts = (
        metadata_df
        .groupby("shard")
        .size()
    )

    if len(shard_counts) != len(
        shard_files
    ):

        raise RuntimeError(
            "Number of metadata shards "
            "does not match NPZ shard count."
        )

    print(
        "Shard coverage       : OK"
    )

    # ========================================================
    # SAFETY CHECK 7
    # Start/end frame validity
    # ========================================================

    invalid_metadata_ranges = int(
        (
            metadata_df[
                "start_frame"
            ]
            >
            metadata_df[
                "end_frame"
            ]
        )
        .sum()
    )

    print(
        f"Invalid frame ranges : "
        f"{invalid_metadata_ranges}"
    )

    if invalid_metadata_ranges > 0:

        raise RuntimeError(
            "Invalid frame ranges "
            "detected in metadata."
        )

    # ========================================================
    # WRITE TEMPORARY METADATA
    #
    # Existing metadata is STILL untouched.
    # ========================================================

    print()
    print(
        "Writing temporary metadata..."
    )

    metadata_df.to_csv(
        temp_metadata_path,
        index=False
    )

    print(
        f"Temporary file:"
    )

    print(
        f"  {temp_metadata_path}"
    )

    # ========================================================
    # VERIFY TEMPORARY CSV
    # ========================================================

    verify_df = pd.read_csv(
        temp_metadata_path
    )

    if len(verify_df) != (
        total_shard_clips
    ):

        # Delete only our temporary file
        if temp_metadata_path.exists():
            temp_metadata_path.unlink()

        raise RuntimeError(
            "Temporary metadata "
            "verification failed."
        )

    verify_ids = (
        verify_df[
            "clip_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    if not np.array_equal(
        verify_ids,
        expected_global_ids
    ):

        if temp_metadata_path.exists():
            temp_metadata_path.unlink()

        raise RuntimeError(
            "Temporary metadata "
            "global ID verification failed."
        )

    print(
        "Temporary metadata verification : OK"
    )

    # ========================================================
    # FINAL REPLACEMENT
    #
    # Only NOW replace old metadata.
    # ========================================================

    print()
    print(
        "All safety checks passed."
    )

    print(
        "Installing repaired metadata..."
    )

    # --------------------------------------------------------
    # If old metadata exists:
    # replace it with temporary file.
    #
    # Backup remains safely available.
    # --------------------------------------------------------

    temp_metadata_path.replace(
        metadata_path
    )

    # ========================================================
    # FINAL FILE CHECK
    # ========================================================

    if not metadata_path.exists():

        raise RuntimeError(
            "Final metadata file "
            "was not created."
        )

    final_df = pd.read_csv(
        metadata_path
    )

    if len(final_df) != (
        total_shard_clips
    ):

        raise RuntimeError(
            "Final metadata row count "
            "verification failed."
        )

    # ========================================================
    # SUCCESS
    # ========================================================

    print()
    print("=" * 70)
    print(
        f"PHASE 6 REPAIR v3 SAFE "
        f"COMPLETED : {video_id}"
    )
    print("=" * 70)

    print(
        f"Metadata rows     : "
        f"{len(final_df)}"
    )

    print(
        f"Shard clips       : "
        f"{total_shard_clips}"
    )

    print(
        f"Global ID range   : "
        f"0..{len(final_df) - 1}"
    )

    print(
        f"Duplicate IDs     : "
        f"{duplicate_global_ids}"
    )

    print(
        f"NPZ shards read   : "
        f"{len(shard_files)}"
    )

    print(
        "clip_npy modified : NO"
    )

    print(
        f"Metadata installed: "
        f"{metadata_path}"
    )

    print("=" * 70)

    return {

        "video_id":
            video_id,

        "status":
            "REPAIRED",

        "metadata_rows":
            len(final_df),

        "shard_clips":
            total_shard_clips,

        "shard_files":
            len(shard_files),

        "duplicate_ids":
            duplicate_global_ids,

        "metadata_path":
            str(metadata_path),

    }


# ============================================================
# RUN REPAIR
# ============================================================

PHASE6_REPAIR_V3_SUMMARY = []


for video_id in REPAIR_VIDEO_IDS:

    try:

        result = repair_metadata_v3_safe(
            video_id
        )

        PHASE6_REPAIR_V3_SUMMARY.append(
            result
        )

    except Exception as e:

        print()
        print("=" * 70)
        print(
            f"❌ REPAIR FAILED : "
            f"{video_id}"
        )

        print(
            f"Error: "
            f"{type(e).__name__}: {e}"
        )

        traceback.print_exc()

        PHASE6_REPAIR_V3_SUMMARY.append({

            "video_id":
                video_id,

            "status":
                "FAILED",

            "metadata_rows":
                0,

            "shard_clips":
                0,

            "shard_files":
                0,

            "duplicate_ids":
                None,

            "metadata_path":
                None,

            "error":
                f"{type(e).__name__}: {e}",

        })


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 70)
print(
    "PHASE 6 — REPAIR v3 SAFE SUMMARY"
)
print("=" * 70)

repair_summary_df = pd.DataFrame(
    PHASE6_REPAIR_V3_SUMMARY
)

if not repair_summary_df.empty:

    print(
        repair_summary_df.to_string(
            index=False
        )
    )

print("=" * 70)
print(
    "CELL 17B COMPLETE"
)
print("=" * 70)

PHASE 6 — REPAIR METADATA v3 SAFE

REPAIR v3 SAFE : video001
Output    : C:\LipReadingSSL\output\video001
Shard dir : C:\LipReadingSSL\output\video001\clip_npy
Metadata  : C:\LipReadingSSL\output\video001\clip_metadata.csv

NPZ shards found : 79

Backup created by COPY:
  C:\LipReadingSSL\output\video001\backups\clip_metadata_before_repair_v3\clip_metadata_20260829_022332.csv
[  1/79] shard_000000.npz     clips=512  global=0..511
[  2/79] shard_000001.npz     clips=512  global=512..1023
[  3/79] shard_000002.npz     clips=512  global=1024..1535
[  4/79] shard_000003.npz     clips=512  global=1536..2047
[  5/79] shard_000004.npz     clips=512  global=2048..2559
[  6/79] shard_000005.npz     clips=512  global=2560..3071
[  7/79] shard_000006.npz     clips=512  global=3072..3583
[  8/79] shard_000007.npz     clips=512  global=3584..4095
[  9/79] shard_000008.npz     clips=512  global=4096..4607
[ 10/79] shard_000009.npz     clips=512  global=4608..5119
[ 11/79] shard_000010.npz     clips=

In [34]:
# ============================================================
# PHASE 6 — CELL 17C
# REPAIR VIDEO003 METADATA v3 SAFE
#
# PURPOSE:
#   Repair ONLY video003 metadata.
#
# SOURCE OF TRUTH:
#   Every .npz file inside:
#       output/video003/clip_npy
#
# IMPORTANT:
#   - DOES NOT MODIFY clip_npy
#   - DOES NOT MODIFY any .npz file
#   - DOES NOT regenerate clips
#   - DOES NOT trust old clip_metadata.csv
#   - Reads EVERY .npz shard
#   - Converts LOCAL shard indices -> GLOBAL clip_id
#   - Creates backup before overwrite
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import numpy as np
import pandas as pd
import traceback


print("=" * 70)
print("PHASE 6 — REPAIR VIDEO003 METADATA v3 SAFE")
print("=" * 70)


# ============================================================
# Target video
# ============================================================

video_id = "video003"


# ============================================================
# Paths
# ============================================================

output_dir = (
    PROJECT_ROOT
    / "output"
    / video_id
)

shard_dir = (
    output_dir
    / "clip_npy"
)

metadata_path = (
    output_dir
    / "clip_metadata.csv"
)

backup_dir = (
    output_dir
    / "backups"
    / "clip_metadata_before_repair_v3"
    / "video003"
)


print()
print(f"Output    : {output_dir}")
print(f"Shard dir : {shard_dir}")
print(f"Metadata  : {metadata_path}")


# ============================================================
# Validate paths
# ============================================================

if not output_dir.exists():

    raise FileNotFoundError(
        f"Output directory not found:\n"
        f"{output_dir}"
    )


if not shard_dir.exists():

    raise FileNotFoundError(
        f"Shard directory not found:\n"
        f"{shard_dir}"
    )


# ============================================================
# Find EVERY NPZ shard
# ============================================================

shard_files = sorted(
    shard_dir.glob("*.npz")
)


print()
print(
    f"NPZ shards found : "
    f"{len(shard_files)}"
)


if not shard_files:

    raise RuntimeError(
        "No .npz shard files found."
    )


# ============================================================
# Backup old metadata
# ============================================================

if metadata_path.exists():

    backup_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    backup_path = (
        backup_dir
        /
        f"clip_metadata_{timestamp}.csv"
    )

    counter = 1

    while backup_path.exists():

        backup_path = (
            backup_dir
            /
            (
                f"clip_metadata_"
                f"{timestamp}_"
                f"{counter}.csv"
            )
        )

        counter += 1

    shutil.copy2(
        metadata_path,
        backup_path
    )

    print()
    print(
        "Backup created by COPY:"
    )

    print(
        f"  {backup_path}"
    )

else:

    backup_path = None

    print()
    print(
        "No existing metadata found."
    )


# ============================================================
# Read every shard
# ============================================================

metadata_records = []

global_clip_id = 0
total_shard_clips = 0


print()


for shard_number, shard_path in enumerate(
    shard_files,
    start=1
):

    try:

        # ----------------------------------------------------
        # Read NPZ
        # ----------------------------------------------------

        with np.load(
            shard_path,
            allow_pickle=False
        ) as data:

            # ------------------------------------------------
            # Required arrays
            # ------------------------------------------------

            required_keys = [
                "clips",
                "start_frames",
                "end_frames",
                "clip_indices",
            ]

            missing_keys = [
                key
                for key in required_keys
                if key not in data.files
            ]

            if missing_keys:

                raise RuntimeError(
                    f"{shard_path.name}: "
                    f"missing keys "
                    f"{missing_keys}"
                )


            clips = data["clips"]

            start_frames = data[
                "start_frames"
            ]

            end_frames = data[
                "end_frames"
            ]

            clip_indices = data[
                "clip_indices"
            ]


            # ------------------------------------------------
            # Number of clips
            # ------------------------------------------------

            n_clips = len(
                clip_indices
            )


            # ------------------------------------------------
            # Shape consistency
            # ------------------------------------------------

            if len(start_frames) != n_clips:

                raise RuntimeError(
                    f"{shard_path.name}: "
                    f"start_frames count "
                    f"does not match "
                    f"clip_indices count"
                )


            if len(end_frames) != n_clips:

                raise RuntimeError(
                    f"{shard_path.name}: "
                    f"end_frames count "
                    f"does not match "
                    f"clip_indices count"
                )


            if len(clips) != n_clips:

                raise RuntimeError(
                    f"{shard_path.name}: "
                    f"clips count "
                    f"does not match "
                    f"clip_indices count"
                )


            # ------------------------------------------------
            # Validate local indices
            # ------------------------------------------------

            local_indices = np.asarray(
                clip_indices,
                dtype=np.int64
            )


            expected_local_indices = np.arange(
                n_clips,
                dtype=np.int64
            )


            if not np.array_equal(
                local_indices,
                expected_local_indices
            ):

                raise RuntimeError(
                    f"{shard_path.name}: "
                    f"clip_indices are not "
                    f"local sequential indices "
                    f"0..{n_clips - 1}"
                )


            # ------------------------------------------------
            # Build metadata records
            # ------------------------------------------------

            for local_index in range(
                n_clips
            ):

                record = {

                    "clip_id":
                        int(
                            global_clip_id
                            + local_index
                        ),

                    "video":
                        video_id,

                    "shard":
                        shard_path.name,

                    "shard_index":
                        int(
                            local_index
                        ),

                    "start_frame":
                        int(
                            start_frames[
                                local_index
                            ]
                        ),

                    "end_frame":
                        int(
                            end_frames[
                                local_index
                            ]
                        ),

                }

                metadata_records.append(
                    record
                )


            # ------------------------------------------------
            # Update counters
            # ------------------------------------------------

            shard_start_global = (
                global_clip_id
            )

            global_clip_id += n_clips

            total_shard_clips += n_clips

            shard_end_global = (
                global_clip_id - 1
            )


            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            print(
                f"[{shard_number:>3}/"
                f"{len(shard_files)}] "
                f"{shard_path.name:<20} "
                f"clips={n_clips:<4} "
                f"global="
                f"{shard_start_global}"
                f".."
                f"{shard_end_global}"
            )


    except Exception:

        print()
        print(
            f"❌ FAILED reading "
            f"{shard_path.name}"
        )

        raise


# ============================================================
# Build DataFrame
# ============================================================

metadata_df = pd.DataFrame(
    metadata_records
)


# ============================================================
# Safety checks BEFORE writing
# ============================================================

print()
print("=" * 70)
print("REPAIR v3 SAFE — SAFETY CHECKS")
print("=" * 70)


print(
    f"Shard files       : "
    f"{len(shard_files)}"
)

print(
    f"Shard clips       : "
    f"{total_shard_clips}"
)

print(
    f"Metadata records  : "
    f"{len(metadata_df)}"
)


# ------------------------------------------------------------
# Check count
# ------------------------------------------------------------

if len(metadata_df) != total_shard_clips:

    raise RuntimeError(
        "Metadata record count does not "
        "match total shard clips."
    )


# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

required_columns = [
    "clip_id",
    "video",
    "shard",
    "shard_index",
    "start_frame",
    "end_frame",
]


missing_columns = [
    column
    for column in required_columns
    if column not in metadata_df.columns
]


if missing_columns:

    raise RuntimeError(
        f"Missing metadata columns: "
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# Duplicate global IDs
# ------------------------------------------------------------

duplicate_global_ids = int(
    metadata_df[
        "clip_id"
    ]
    .duplicated()
    .sum()
)


print(
    f"Duplicate global IDs : "
    f"{duplicate_global_ids}"
)


if duplicate_global_ids > 0:

    raise RuntimeError(
        "Duplicate GLOBAL clip IDs detected."
    )


# ------------------------------------------------------------
# Global ID continuity
# ------------------------------------------------------------

expected_global_ids = np.arange(
    len(metadata_df),
    dtype=np.int64
)


actual_global_ids = (
    metadata_df[
        "clip_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    actual_global_ids,
    expected_global_ids
):

    raise RuntimeError(
        "Global clip IDs are not continuous "
        "from 0."
    )


print(
    "Global ID continuity : OK"
)


# ------------------------------------------------------------
# Video consistency
# ------------------------------------------------------------

video_values = set(
    metadata_df[
        "video"
    ]
    .astype(str)
    .unique()
)


if video_values != {
    video_id
}:

    raise RuntimeError(
        "Video ID mismatch detected."
    )


print(
    "Video consistency    : OK"
)


# ------------------------------------------------------------
# Frame range validation
# ------------------------------------------------------------

numeric_start = pd.to_numeric(
    metadata_df[
        "start_frame"
    ],
    errors="coerce"
)

numeric_end = pd.to_numeric(
    metadata_df[
        "end_frame"
    ],
    errors="coerce"
)


invalid_frame_rows = int(
    (
        numeric_start.isna()
        |
        numeric_end.isna()
        |
        (numeric_start > numeric_end)
    ).sum()
)


print(
    f"Invalid frame ranges : "
    f"{invalid_frame_rows}"
)


if invalid_frame_rows > 0:

    raise RuntimeError(
        "Invalid frame ranges detected."
    )


# ============================================================
# Shard index validation
# ============================================================

invalid_shard_index = 0


for shard_path in shard_files:

    shard_name = shard_path.name

    rows = metadata_df[
        metadata_df[
            "shard"
        ] == shard_name
    ]

    expected_indices = np.arange(
        len(rows),
        dtype=np.int64
    )

    actual_indices = (
        rows[
            "shard_index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    if not np.array_equal(
        actual_indices,
        expected_indices
    ):

        invalid_shard_index += 1


print(
    f"Shard index errors   : "
    f"{invalid_shard_index}"
)


if invalid_shard_index > 0:

    raise RuntimeError(
        "Shard-local indices are invalid."
    )


# ============================================================
# WRITE METADATA
# ============================================================

print()
print("=" * 70)
print("WRITING REPAIRED METADATA")
print("=" * 70)


metadata_df.to_csv(
    metadata_path,
    index=False
)


# ============================================================
# Verify written metadata
# ============================================================

if not metadata_path.exists():

    raise RuntimeError(
        "Metadata file was not created."
    )


written_df = pd.read_csv(
    metadata_path,
    low_memory=False
)


# ------------------------------------------------------------
# Row count
# ------------------------------------------------------------

if len(written_df) != total_shard_clips:

    raise RuntimeError(
        "Written metadata row count does "
        "not match shard count."
    )


# ------------------------------------------------------------
# Columns
# ------------------------------------------------------------

if list(
    written_df.columns
) != required_columns:

    raise RuntimeError(
        "Written metadata columns do not "
        "match expected schema."
    )


# ------------------------------------------------------------
# IDs
# ------------------------------------------------------------

written_ids = (
    written_df[
        "clip_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    written_ids,
    expected_global_ids
):

    raise RuntimeError(
        "Written GLOBAL clip IDs are invalid."
    )


# ============================================================
# SUCCESS
# ============================================================

print()
print("=" * 70)
print(
    "PHASE 6 REPAIR VIDEO003 v3 SAFE COMPLETED"
)
print("=" * 70)


print(
    f"Video             : {video_id}"
)

print(
    f"Metadata rows     : "
    f"{len(written_df)}"
)

print(
    f"Shard files       : "
    f"{len(shard_files)}"
)

print(
    f"Shard clips       : "
    f"{total_shard_clips}"
)

print(
    f"Global ID range   : "
    f"0..{len(written_df) - 1}"
)

print(
    f"Duplicate IDs     : "
    f"{duplicate_global_ids}"
)

print(
    f"Invalid ranges    : "
    f"{invalid_frame_rows}"
)

print(
    f"Metadata          : "
    f"{metadata_path}"
)

print(
    "clip_npy modified : NO"
)

print("=" * 70)


# ============================================================
# Final result
# ============================================================

PHASE6_VIDEO003_REPAIR_RESULT = {

    "video_id":
        video_id,

    "status":
        "REPAIRED",

    "metadata_rows":
        len(written_df),

    "shard_files":
        len(shard_files),

    "shard_clips":
        total_shard_clips,

    "duplicate_ids":
        duplicate_global_ids,

    "invalid_frame_ranges":
        invalid_frame_rows,

    "metadata_path":
        str(metadata_path),

    "backup_path":
        (
            str(backup_path)
            if backup_path is not None
            else None
        ),

    "clip_npy_modified":
        False,

}


print()
print(
    "CELL 17C COMPLETE"
)

PHASE 6 — REPAIR VIDEO003 METADATA v3 SAFE

Output    : C:\LipReadingSSL\output\video003
Shard dir : C:\LipReadingSSL\output\video003\clip_npy
Metadata  : C:\LipReadingSSL\output\video003\clip_metadata.csv

NPZ shards found : 52

Backup created by COPY:
  C:\LipReadingSSL\output\video003\backups\clip_metadata_before_repair_v3\video003\clip_metadata_20260829_174015.csv

[  1/52] shard_000000.npz     clips=512  global=0..511
[  2/52] shard_000001.npz     clips=512  global=512..1023
[  3/52] shard_000002.npz     clips=512  global=1024..1535
[  4/52] shard_000003.npz     clips=512  global=1536..2047
[  5/52] shard_000004.npz     clips=512  global=2048..2559
[  6/52] shard_000005.npz     clips=512  global=2560..3071
[  7/52] shard_000006.npz     clips=512  global=3072..3583
[  8/52] shard_000007.npz     clips=512  global=3584..4095
[  9/52] shard_000008.npz     clips=512  global=4096..4607
[ 10/52] shard_000009.npz     clips=512  global=4608..5119
[ 11/52] shard_000010.npz     clips=512  gl

In [35]:
# ============================================================
# PHASE 6 — CELL 18
# FINAL STRUCTURAL VALIDATION v4
#
# Run AFTER:
#   Cell 17B — Repair Metadata v3 Safe
#
# IMPORTANT
# ------------------------------------------------------------
# READ ONLY
#
# This cell:
#   - reads clip_metadata.csv
#   - reads EVERY .npz shard
#   - does NOT modify clip_npy
#   - does NOT modify clip_metadata.csv
#   - does NOT repair anything
#
# Validation model:
#
#   metadata
#       |
#       +-- shard ---------> shard_XXXXXX.npz
#       |
#       +-- shard_index ---> LOCAL clip position inside shard
#       |
#       +-- start_frame
#       +-- end_frame
#
# We DO NOT assume:
#
#   clip_id == 0..N-1
#
# We DO NOT treat clip_indices from different shards as
# globally unique.
#
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import traceback


print("=" * 70)
print("PHASE 6 — FINAL STRUCTURAL VALIDATION v4")
print("=" * 70)


# ============================================================
# VIDEO LIST
# ============================================================

VALIDATE_VIDEO_IDS = [
    "video001",
    "video002",
    "video003",
]


# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate_phase6_final_v4(video_id):

    print()
    print("=" * 70)
    print(f"FINAL VALIDATION : {video_id}")
    print("=" * 70)

    # ========================================================
    # PATHS
    # ========================================================

    output_dir = (
        PROJECT_ROOT
        / "output"
        / video_id
    )

    metadata_path = (
        output_dir
        / "clip_metadata.csv"
    )

    shard_dir = (
        output_dir
        / "clip_npy"
    )

    print(
        f"Output   : {output_dir}"
    )

    print(
        f"Metadata : {metadata_path}"
    )

    print(
        f"Shards   : {shard_dir}"
    )

    # ========================================================
    # PATH CHECK
    # ========================================================

    if not output_dir.exists():

        raise FileNotFoundError(
            f"Output directory does not exist:\n"
            f"{output_dir}"
        )

    if not metadata_path.exists():

        raise FileNotFoundError(
            f"Metadata file does not exist:\n"
            f"{metadata_path}"
        )

    if not shard_dir.exists():

        raise FileNotFoundError(
            f"Shard directory does not exist:\n"
            f"{shard_dir}"
        )

    # ========================================================
    # READ METADATA
    # ========================================================

    metadata_df = pd.read_csv(
        metadata_path,
        low_memory=False
    )

    metadata_rows = len(
        metadata_df
    )

    print()
    print(
        f"Metadata rows : "
        f"{metadata_rows}"
    )

    print(
        "Metadata columns:"
    )

    for column in metadata_df.columns:

        print(
            f"  - {column}"
        )

    # ========================================================
    # REQUIRED COLUMNS
    # ========================================================

    required_columns = [
        "clip_id",
        "video",
        "shard",
        "shard_index",
        "start_frame",
        "end_frame",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in metadata_df.columns
    ]

    if missing_columns:

        print()
        print(
            "⚠️ Missing metadata columns:"
        )

        for column in missing_columns:

            print(
                f"   - {column}"
            )

    full_schema_available = (
        len(missing_columns) == 0
    )

    # ========================================================
    # CLIP ID CHECK
    #
    # Only uniqueness/null values are checked.
    #
    # No sequential-number assumption.
    # ========================================================

    duplicate_clip_ids = 0
    null_clip_ids = 0

    if "clip_id" in metadata_df.columns:

        duplicate_clip_ids = int(
            metadata_df[
                "clip_id"
            ]
            .duplicated()
            .sum()
        )

        null_clip_ids = int(
            metadata_df[
                "clip_id"
            ]
            .isna()
            .sum()
        )

    print(
        f"Duplicate clip IDs : "
        f"{duplicate_clip_ids}"
    )

    print(
        f"Null clip IDs      : "
        f"{null_clip_ids}"
    )

    clip_id_valid = (
        duplicate_clip_ids == 0
        and
        null_clip_ids == 0
    )

    # ========================================================
    # METADATA FRAME RANGE CHECK
    # ========================================================

    invalid_metadata_ranges = 0

    if (
        "start_frame" in metadata_df.columns
        and
        "end_frame" in metadata_df.columns
    ):

        metadata_start = pd.to_numeric(
            metadata_df[
                "start_frame"
            ],
            errors="coerce"
        )

        metadata_end = pd.to_numeric(
            metadata_df[
                "end_frame"
            ],
            errors="coerce"
        )

        invalid_metadata_ranges = int(
            (
                metadata_start.isna()
                |
                metadata_end.isna()
                |
                (metadata_start > metadata_end)
                |
                (metadata_start < 0)
                |
                (metadata_end < 0)
            )
            .sum()
        )

    print(
        f"Invalid metadata frame ranges : "
        f"{invalid_metadata_ranges}"
    )

    metadata_frame_ranges_valid = (
        invalid_metadata_ranges == 0
    )

    # ========================================================
    # VIDEO COLUMN CHECK
    # ========================================================

    video_consistent = True

    unexpected_video_values = []

    if "video" in metadata_df.columns:

        metadata_video_values = set(
            metadata_df[
                "video"
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
        )

        allowed_video_values = {
            str(video_id),
            f"{video_id}.webm",
        }

        unexpected_video_values = sorted(
            metadata_video_values
            -
            allowed_video_values
        )

        if unexpected_video_values:

            video_consistent = False

            print(
                "⚠️ Unexpected video values:"
            )

            for value in unexpected_video_values:

                print(
                    f"   - {value}"
                )

    print(
        f"Video consistency : "
        f"{'OK' if video_consistent else 'FAILED'}"
    )

    # ========================================================
    # FIND ALL NPZ SHARDS
    # ========================================================

    shard_files = sorted(
        shard_dir.glob("*.npz")
    )

    if not shard_files:

        raise RuntimeError(
            "No .npz shard files found."
        )

    print()
    print(
        f"Shard files : "
        f"{len(shard_files)}"
    )

    # ========================================================
    # SHARD INFORMATION
    # ========================================================

    shard_info = {}

    total_shard_clips = 0

    shard_integrity_valid = True

    shard_local_indices_valid = True

    shard_frame_ranges_valid = True

    # ========================================================
    # READ EVERY SHARD
    # ========================================================

    for shard_number, shard_path in enumerate(
        shard_files,
        start=1
    ):

        try:

            with np.load(
                shard_path,
                allow_pickle=False
            ) as data:

                required_npz_keys = [
                    "clips",
                    "start_frames",
                    "end_frames",
                    "clip_indices",
                ]

                missing_npz_keys = [
                    key
                    for key in required_npz_keys
                    if key not in data.files
                ]

                if missing_npz_keys:

                    shard_integrity_valid = False

                    print(
                        f"[{shard_number:>3}/"
                        f"{len(shard_files)}] "
                        f"{shard_path.name:<20} "
                        f"❌ missing keys: "
                        f"{missing_npz_keys}"
                    )

                    continue

                # ------------------------------------------------
                # READ ARRAYS
                # ------------------------------------------------

                clips = data[
                    "clips"
                ]

                start_frames = np.asarray(
                    data[
                        "start_frames"
                    ]
                )

                end_frames = np.asarray(
                    data[
                        "end_frames"
                    ]
                )

                clip_indices = np.asarray(
                    data[
                        "clip_indices"
                    ]
                )

                n_clips = len(
                    clips
                )

                # ------------------------------------------------
                # ARRAY LENGTHS
                # ------------------------------------------------

                lengths_match = (
                    len(start_frames)
                    ==
                    n_clips
                    and
                    len(end_frames)
                    ==
                    n_clips
                    and
                    len(clip_indices)
                    ==
                    n_clips
                )

                if not lengths_match:

                    shard_integrity_valid = False

                    print(
                        f"[{shard_number:>3}/"
                        f"{len(shard_files)}] "
                        f"{shard_path.name:<20} "
                        f"❌ array length mismatch"
                    )

                    continue

                # ------------------------------------------------
                # LOCAL INDEX VALIDATION
                #
                # IMPORTANT:
                #
                # clip_indices are LOCAL to each shard.
                #
                # Therefore:
                #
                # shard_000000:
                #     0..511
                #
                # shard_000001:
                #     0..511
                #
                # is VALID.
                # ------------------------------------------------

                expected_local_indices = np.arange(
                    n_clips,
                    dtype=clip_indices.dtype
                )

                local_indices_valid = np.array_equal(
                    clip_indices,
                    expected_local_indices
                )

                if not local_indices_valid:

                    shard_local_indices_valid = False

                # ------------------------------------------------
                # SHARD FRAME RANGE VALIDATION
                # ------------------------------------------------

                invalid_shard_ranges = int(
                    (
                        start_frames
                        >
                        end_frames
                    ).sum()
                )

                if invalid_shard_ranges > 0:

                    shard_frame_ranges_valid = False

                # ------------------------------------------------
                # STORE INFORMATION
                # ------------------------------------------------

                shard_info[
                    shard_path.name
                ] = {

                    "path":
                        shard_path,

                    "shard_number":
                        shard_number,

                    "clip_count":
                        n_clips,

                    "local_indices":
                        clip_indices,

                    "start_frames":
                        start_frames,

                    "end_frames":
                        end_frames,

                    "local_indices_valid":
                        local_indices_valid,

                    "invalid_ranges":
                        invalid_shard_ranges,
                }

                total_shard_clips += (
                    n_clips
                )

                # ------------------------------------------------
                # PRINT
                # ------------------------------------------------

                if n_clips > 0:

                    first_local = int(
                        clip_indices[0]
                    )

                    last_local = int(
                        clip_indices[-1]
                    )

                else:

                    first_local = None
                    last_local = None

                if (
                    local_indices_valid
                    and
                    invalid_shard_ranges == 0
                ):

                    symbol = "✅"

                else:

                    symbol = "⚠️"

                print(
                    f"[{shard_number:>3}/"
                    f"{len(shard_files)}] "
                    f"{shard_path.name:<20} "
                    f"clips={n_clips:<4} "
                    f"local="
                    f"{first_local}"
                    f".."
                    f"{last_local} "
                    f"{symbol}"
                )

        except Exception as shard_error:

            shard_integrity_valid = False

            print()
            print(
                f"[{shard_number:>3}/"
                f"{len(shard_files)}] "
                f"{shard_path.name}"
            )

            print(
                f"❌ Shard read error: "
                f"{type(shard_error).__name__}: "
                f"{shard_error}"
            )

    # ========================================================
    # SHARD SUMMARY
    # ========================================================

    print()
    print(
        f"Total shard clips : "
        f"{total_shard_clips}"
    )

    # ========================================================
    # METADATA ↔ SHARD COUNT
    # ========================================================

    metadata_shard_count_match = (
        metadata_rows
        ==
        total_shard_clips
    )

    if metadata_shard_count_match:

        print(
            "✅ Metadata ↔ shard count matched"
        )

    else:

        print(
            "❌ Metadata ↔ shard count mismatch"
        )

        print(
            f"   Metadata rows : "
            f"{metadata_rows}"
        )

        print(
            f"   Shard clips   : "
            f"{total_shard_clips}"
        )

    # ========================================================
    # BUILD EXPECTED SHARD/LOCAL INDEX MAP
    #
    # This is the correct mapping model.
    #
    # Key:
    #
    #     (shard_name, local_index)
    #
    # Each key should identify exactly one physical clip.
    # ========================================================

    physical_clip_keys = set()

    duplicate_physical_keys = 0

    for shard_name, info in shard_info.items():

        local_indices = info[
            "local_indices"
        ]

        for local_index in local_indices:

            key = (
                shard_name,
                int(local_index)
            )

            if key in physical_clip_keys:

                duplicate_physical_keys += 1

            else:

                physical_clip_keys.add(
                    key
                )

    print(
        f"Duplicate physical "
        f"(shard, local_index) keys : "
        f"{duplicate_physical_keys}"
    )

    physical_keys_valid = (
        duplicate_physical_keys == 0
    )

    # ========================================================
    # METADATA ↔ PHYSICAL SHARD MAPPING
    #
    # Here shard_index means LOCAL POSITION inside shard.
    # ========================================================

    metadata_mapping_valid = True

    metadata_mapping_checked = False

    metadata_mapping_missing = 0

    metadata_mapping_out_of_range = 0

    metadata_mapping_unknown_shard = 0

    metadata_mapping_duplicate = 0

    metadata_physical_keys = set()

    if (
        "shard" in metadata_df.columns
        and
        "shard_index" in metadata_df.columns
    ):

        metadata_mapping_checked = True

        metadata_shard_series = (
            metadata_df[
                "shard"
            ]
            .astype(str)
            .str.strip()
        )

        metadata_index_series = pd.to_numeric(
            metadata_df[
                "shard_index"
            ],
            errors="coerce"
        )

        for row_number in range(
            len(metadata_df)
        ):

            shard_name = (
                metadata_shard_series.iloc[
                    row_number
                ]
            )

            local_index_value = (
                metadata_index_series.iloc[
                    row_number
                ]
            )

            # ------------------------------------------------
            # Unknown shard
            # ------------------------------------------------

            if shard_name not in shard_info:

                metadata_mapping_unknown_shard += 1

                metadata_mapping_valid = False

                continue

            # ------------------------------------------------
            # Invalid shard index
            # ------------------------------------------------

            if pd.isna(
                local_index_value
            ):

                metadata_mapping_out_of_range += 1

                metadata_mapping_valid = False

                continue

            local_index = int(
                local_index_value
            )

            # ------------------------------------------------
            # Check local index range
            # ------------------------------------------------

            shard_clip_count = (
                shard_info[
                    shard_name
                ]["clip_count"]
            )

            if (
                local_index < 0
                or
                local_index >= shard_clip_count
            ):

                metadata_mapping_out_of_range += 1

                metadata_mapping_valid = False

                continue

            # ------------------------------------------------
            # Physical key
            # ------------------------------------------------

            physical_key = (
                shard_name,
                local_index
            )

            # ------------------------------------------------
            # Check duplicate metadata mapping
            # ------------------------------------------------

            if physical_key in (
                metadata_physical_keys
            ):

                metadata_mapping_duplicate += 1

                metadata_mapping_valid = False

            else:

                metadata_physical_keys.add(
                    physical_key
                )

    else:

        print()
        print(
            "⚠️ Metadata ↔ shard mapping "
            "not checked because "
            "shard/shard_index columns "
            "are unavailable."
        )

    # ========================================================
    # PRINT MAPPING RESULT
    # ========================================================

    if metadata_mapping_checked:

        print()

        print(
            "Metadata ↔ shard/local mapping"
        )

        print(
            f"   Unknown shard       : "
            f"{metadata_mapping_unknown_shard}"
        )

        print(
            f"   Out-of-range index  : "
            f"{metadata_mapping_out_of_range}"
        )

        print(
            f"   Duplicate mappings  : "
            f"{metadata_mapping_duplicate}"
        )

        if metadata_mapping_valid:

            print(
                "   ✅ Mapping structure valid"
            )

        else:

            print(
                "   ❌ Mapping structure invalid"
            )

    # ========================================================
    # IMPORTANT:
    #
    # DO NOT REQUIRE:
    #
    # metadata physical keys == all shard keys
    #
    # if metadata row count already matches shard count,
    # then compare sets explicitly.
    # ========================================================

    metadata_missing_physical_keys = 0

    shard_missing_metadata_keys = 0

    physical_mapping_complete = True

    if metadata_mapping_checked:

        metadata_missing_physical_keys = len(
            physical_clip_keys
            -
            metadata_physical_keys
        )

        shard_missing_metadata_keys = len(
            metadata_physical_keys
            -
            physical_clip_keys
        )

        if (
            metadata_missing_physical_keys == 0
            and
            shard_missing_metadata_keys == 0
        ):

            print(
                "   ✅ Metadata ↔ physical "
                "(shard, local_index) keys matched"
            )

        else:

            physical_mapping_complete = False

            print(
                "   ❌ Metadata ↔ physical "
                "clip keys mismatch"
            )

            print(
                f"      Shard clips missing "
                f"from metadata : "
                f"{metadata_missing_physical_keys}"
            )

            print(
                f"      Metadata clips not "
                f"in shards       : "
                f"{shard_missing_metadata_keys}"
            )

    # ========================================================
    # FRAME CONSISTENCY BETWEEN METADATA AND NPZ
    #
    # Compare:
    #
    # metadata start_frame/end_frame
    #
    # against:
    #
    # shard start_frames/end_frames
    #
    # using:
    #
    # (shard, shard_index)
    #
    # ========================================================

    frame_mapping_checked = False

    frame_mapping_valid = True

    frame_mismatches = 0

    if (
        metadata_mapping_checked
        and
        "start_frame" in metadata_df.columns
        and
        "end_frame" in metadata_df.columns
    ):

        frame_mapping_checked = True

        metadata_start_series = pd.to_numeric(
            metadata_df[
                "start_frame"
            ],
            errors="coerce"
        )

        metadata_end_series = pd.to_numeric(
            metadata_df[
                "end_frame"
            ],
            errors="coerce"
        )

        for row_number in range(
            len(metadata_df)
        ):

            shard_name = (
                metadata_shard_series.iloc[
                    row_number
                ]
            )

            local_index_value = (
                metadata_index_series.iloc[
                    row_number
                ]
            )

            if shard_name not in shard_info:

                continue

            if pd.isna(
                local_index_value
            ):

                continue

            local_index = int(
                local_index_value
            )

            info = shard_info[
                shard_name
            ]

            if (
                local_index < 0
                or
                local_index >= info[
                    "clip_count"
                ]
            ):

                continue

            npz_start = int(
                info[
                    "start_frames"
                ][local_index]
            )

            npz_end = int(
                info[
                    "end_frames"
                ][local_index]
            )

            metadata_start_value = (
                metadata_start_series.iloc[
                    row_number
                ]
            )

            metadata_end_value = (
                metadata_end_series.iloc[
                    row_number
                ]
            )

            if (
                pd.isna(
                    metadata_start_value
                )
                or
                pd.isna(
                    metadata_end_value
                )
            ):

                continue

            if (
                int(metadata_start_value)
                !=
                npz_start
                or
                int(metadata_end_value)
                !=
                npz_end
            ):

                frame_mismatches += 1

        if frame_mismatches > 0:

            frame_mapping_valid = False

    # ========================================================
    # FRAME MAPPING RESULT
    # ========================================================

    if frame_mapping_checked:

        print()

        print(
            f"Metadata ↔ NPZ frame "
            f"mismatches : "
            f"{frame_mismatches}"
        )

        if frame_mapping_valid:

            print(
                "✅ Metadata frame ranges "
                "match NPZ frame ranges"
            )

        else:

            print(
                "❌ Metadata frame ranges "
                "do not match NPZ"
            )

    # ========================================================
    # SHARD LOCAL INDEX RESULT
    # ========================================================

    print()

    if shard_local_indices_valid:

        print(
            "✅ All shard local indices valid"
        )

    else:

        print(
            "❌ Some shard local indices invalid"
        )

    # ========================================================
    # SHARD FRAME RESULT
    # ========================================================

    if shard_frame_ranges_valid:

        print(
            "✅ All NPZ frame ranges valid"
        )

    else:

        print(
            "❌ NPZ frame ranges invalid"
        )

    # ========================================================
    # FINAL PASS CONDITION
    # ========================================================

    final_pass = (

        # Metadata
        metadata_shard_count_match

        and

        # NPZ integrity
        shard_integrity_valid

        and

        # Local clip indices
        shard_local_indices_valid

        and

        # NPZ frame ranges
        shard_frame_ranges_valid

        and

        # Metadata frame ranges
        metadata_frame_ranges_valid

        and

        # clip IDs
        clip_id_valid

        and

        # video
        video_consistent

        and

        # Physical key uniqueness
        physical_keys_valid

        and

        # Metadata mapping
        (
            not metadata_mapping_checked
            or
            metadata_mapping_valid
        )

        and

        # Metadata ↔ physical key set
        (
            not metadata_mapping_checked
            or
            physical_mapping_complete
        )

        and

        # Metadata ↔ NPZ frames
        (
            not frame_mapping_checked
            or
            frame_mapping_valid
        )
    )

    # ========================================================
    # FINAL RESULT
    # ========================================================

    print()

    print("=" * 70)

    if final_pass:

        print(
            "✅ PHASE 6 FINAL VALIDATION PASSED"
        )

    else:

        print(
            "❌ PHASE 6 FINAL VALIDATION FAILED"
        )

    print("=" * 70)

    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {

        "video_id":
            video_id,

        "status":
            "PASSED"
            if final_pass
            else
            "FAILED",

        "metadata_rows":
            metadata_rows,

        "shard_files":
            len(shard_files),

        "shard_clips":
            total_shard_clips,

        "duplicate_clip_ids":
            duplicate_clip_ids,

        "null_clip_ids":
            null_clip_ids,

        "invalid_metadata_ranges":
            invalid_metadata_ranges,

        "metadata_shard_count_match":
            metadata_shard_count_match,

        "shard_integrity_valid":
            shard_integrity_valid,

        "shard_local_indices_valid":
            shard_local_indices_valid,

        "shard_frame_ranges_valid":
            shard_frame_ranges_valid,

        "video_consistent":
            video_consistent,

        "physical_keys_valid":
            physical_keys_valid,

        "metadata_mapping_checked":
            metadata_mapping_checked,

        "metadata_mapping_valid":
            metadata_mapping_valid,

        "metadata_mapping_unknown_shard":
            metadata_mapping_unknown_shard,

        "metadata_mapping_out_of_range":
            metadata_mapping_out_of_range,

        "metadata_mapping_duplicate":
            metadata_mapping_duplicate,

        "physical_mapping_complete":
            physical_mapping_complete,

        "metadata_missing_physical_keys":
            metadata_missing_physical_keys,

        "shard_missing_metadata_keys":
            shard_missing_metadata_keys,

        "frame_mapping_checked":
            frame_mapping_checked,

        "frame_mapping_valid":
            frame_mapping_valid,

        "frame_mismatches":
            frame_mismatches,

        "error":
            None
            if final_pass
            else
            "validation_failed",
    }


# ============================================================
# BATCH VALIDATION
# ============================================================

PHASE6_FINAL_VALIDATION_RESULTS = []


for video_id in VALIDATE_VIDEO_IDS:

    try:

        result = validate_phase6_final_v4(
            video_id
        )

        PHASE6_FINAL_VALIDATION_RESULTS.append(
            result
        )

    except Exception as e:

        print()
        print("=" * 70)
        print(
            f"❌ VALIDATION ERROR : "
            f"{video_id}"
        )
        print("=" * 70)

        print(
            f"{type(e).__name__}: "
            f"{e}"
        )

        traceback.print_exc()

        PHASE6_FINAL_VALIDATION_RESULTS.append({

            "video_id":
                video_id,

            "status":
                "FAILED",

            "metadata_rows":
                None,

            "shard_files":
                None,

            "shard_clips":
                None,

            "duplicate_clip_ids":
                None,

            "null_clip_ids":
                None,

            "invalid_metadata_ranges":
                None,

            "metadata_shard_count_match":
                False,

            "shard_integrity_valid":
                False,

            "shard_local_indices_valid":
                False,

            "shard_frame_ranges_valid":
                False,

            "video_consistent":
                False,

            "physical_keys_valid":
                False,

            "metadata_mapping_checked":
                False,

            "metadata_mapping_valid":
                False,

            "metadata_mapping_unknown_shard":
                None,

            "metadata_mapping_out_of_range":
                None,

            "metadata_mapping_duplicate":
                None,

            "physical_mapping_complete":
                False,

            "metadata_missing_physical_keys":
                None,

            "shard_missing_metadata_keys":
                None,

            "frame_mapping_checked":
                False,

            "frame_mapping_valid":
                False,

            "frame_mismatches":
                None,

            "error":
                f"{type(e).__name__}: {e}",
        })


# ============================================================
# FINAL BATCH SUMMARY
# ============================================================

print()
print("=" * 70)
print(
    "PHASE 6 — FINAL VALIDATION SUMMARY"
)
print("=" * 70)

PHASE6_FINAL_VALIDATION_DF = pd.DataFrame(
    PHASE6_FINAL_VALIDATION_RESULTS
)

if not PHASE6_FINAL_VALIDATION_DF.empty:

    print(
        PHASE6_FINAL_VALIDATION_DF.to_string(
            index=False
        )
    )

print("=" * 70)


# ============================================================
# GO / NO-GO
# ============================================================

all_passed = (
    not PHASE6_FINAL_VALIDATION_DF.empty
    and
    (
        PHASE6_FINAL_VALIDATION_DF[
            "status"
        ]
        ==
        "PASSED"
    ).all()
)

print()

print("=" * 70)

if all_passed:

    print(
        "🟢 PHASE 6 READY FOR PHASE 7"
    )

else:

    print(
        "🔴 PHASE 6 NOT READY FOR PHASE 7"
    )

print("=" * 70)

PHASE 6 — FINAL STRUCTURAL VALIDATION v4

FINAL VALIDATION : video001
Output   : C:\LipReadingSSL\output\video001
Metadata : C:\LipReadingSSL\output\video001\clip_metadata.csv
Shards   : C:\LipReadingSSL\output\video001\clip_npy

Metadata rows : 40076
Metadata columns:
  - clip_id
  - video
  - shard
  - shard_index
  - start_frame
  - end_frame
Duplicate clip IDs : 0
Null clip IDs      : 0
Invalid metadata frame ranges : 0
Video consistency : OK

Shard files : 79
[  1/79] shard_000000.npz     clips=512  local=0..511 ✅
[  2/79] shard_000001.npz     clips=512  local=0..511 ✅
[  3/79] shard_000002.npz     clips=512  local=0..511 ✅
[  4/79] shard_000003.npz     clips=512  local=0..511 ✅
[  5/79] shard_000004.npz     clips=512  local=0..511 ✅
[  6/79] shard_000005.npz     clips=512  local=0..511 ✅
[  7/79] shard_000006.npz     clips=512  local=0..511 ✅
[  8/79] shard_000007.npz     clips=512  local=0..511 ✅
[  9/79] shard_000008.npz     clips=512  local=0..511 ✅
[ 10/79] shard_000009.npz  